### 3.1 Execute SQL Query via Athena (`read_sql`)

In [626]:
import pandas as pd

# 1. Parse timestamps and sort chronologically by lead and time
df11['createdAt_dt'] = pd.to_datetime(df11['createdAt'], errors='coerce')
df_hist = df11.sort_values(by=['lead', 'createdAt_dt']).reset_index(drop=True)

# 2. Shift to get previous stage for each lead transition
df_hist['prev_stage'] = df_hist.groupby('lead')['stage'].shift(1)

# 3. Define Reactivation Conditions:
# A. Lead transitioned OUT of 'Not Interested' into any active stage
mask_ni_transition = (df_hist['prev_stage'] == 'Not Interested') & (df_hist['stage'] != 'Not Interested')

# B. Comment explicitly mentions 'reactivated' (e.g. 'Reactivated by website')
mask_comment_reactivated = df_hist['comments'].astype(str).str.lower().str.contains('reactivated', na=False)

# 4. Filter all reactivation event rows
reactivation_events = df_hist[mask_ni_transition | mask_comment_reactivated].copy()

# 5. Extract unique reactivated lead IDs & reactivation metrics per lead
reactivated_lead_summary = reactivation_events.groupby('lead').agg(
    total_reactivations=('createdAt', 'count'),
    first_reactivation_date=('createdAt_dt', 'min'),
    latest_reactivation_date=('createdAt_dt', 'max'),
    all_reactivation_dates=('createdAt', list)
).reset_index()

# 6. Extract clean list of unique reactivated lead IDs
reactivated_lead_ids = reactivated_lead_summary['lead'].tolist()

print("=" * 75)
print(f"🎉 REACTIVATION ANALYSIS COMPLETE!")
print(f"  • Total Stage Movement Records Analyzed: {len(df11):,}")
print(f"  • Total Unique Reactivated Leads Found:  {len(reactivated_lead_ids):,}")
print("=" * 75)

reactivated_lead_summary.head(10)


🎉 REACTIVATION ANALYSIS COMPLETE!
  • Total Stage Movement Records Analyzed: 3,225,012
  • Total Unique Reactivated Leads Found:  142,764


,lead,total_reactivations,first_reactivation_date,latest_reactivation_date,all_reactivation_dates
0,5faa69fed3cfd05f77b71992,1,2025-10-15 09:44:14.513,2025-10-15 09:44:14.513,[2025-10-15 09:44:14.513000]
1,5fc9f37f3968923070f64bdf,1,2023-07-13 12:03:05.820,2023-07-13 12:03:05.820,[2023-07-13 12:03:05.820000]
2,5fccab56ad3f951c77e56ea6,1,2024-07-18 11:46:03.611,2024-07-18 11:46:03.611,[2024-07-18 11:46:03.611000]
3,5fd1b44aad3f951c77e5b26c,1,2024-11-22 08:35:06.925,2024-11-22 08:35:06.925,[2024-11-22 08:35:06.925000]
4,5fd3c158ad3f951c77e5d379,1,2025-02-25 09:37:49.099,2025-02-25 09:37:49.099,[2025-02-25 09:37:49.099000]
5,5fd45734ad3f951c77e5da93,1,2023-07-23 11:46:04.132,2023-07-23 11:46:04.132,[2023-07-23 11:46:04.132000]
6,5fd9ebff2987d55ff9258bb2,1,2023-09-22 20:23:03.758,2023-09-22 20:23:03.758,[2023-09-22 20:23:03.758000]
7,5fe1aabe0ffd1c4261601e63,1,2023-12-14 06:12:55.499,2023-12-14 06:12:55.499,[2023-12-14 06:12:55.499000]
8,5fe468cb0ffd1c4261607b72,2,2024-09-20 10:07:04.742,2024-12-27 14:59:53.778,"[2024-09-20 10:07:04.742000, 2024-12-27 14:59:..."
9,5fe4b45f0ffd1c4261608ea4,2,2024-09-20 10:06:57.237,2025-04-14 10:26:23.649,"[2024-09-20 10:06:57.237000, 2025-04-14 10:26:..."


isStageMovement
true    3225012
Name: count, dtype: int64[pyarrow]

In [ ]:
# SQL via Athena
df_athena_sql = athena_loader.read_sql("SELECT * FROM leadstagehistories LIMIT 10")
print(f"Lead stage histories (Athena SQL): {len(df_athena_sql)}")
df_athena_sql.head()

### 3.2 Read Table with Date Range Filter via Athena (`read_table`)

In [ ]:
# Read table via Athena
df_athena = athena_loader.read_table(
    "leadstagehistories",
    columns=["_id", "leadId", "stage", "createdAt"],
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
)

print(f"Lead stage histories (Athena Table): {len(df_athena)}")
df_athena.head()

In [ ]:
import re
import json
import boto3
import pandas as pd
import numpy as np
import awswrangler as wr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Initialize AWS Athena Session
session = boto3.Session(region_name="ap-south-1", profile_name="ASBL")
ATHENA_DB = "leadsdb-prod"
S3_OUTPUT = "s3://metabase-asbl/athena-query-results/"

def run_batched_athena_query(sql_template, id_list, id_col_name, batch_size=500):
    """Executes Athena queries in safe 500-ID batches to prevent query payload limit errors."""
    clean_ids = list(set([str(x).replace('.0', '').strip() for x in id_list if pd.notna(x)]))
    batches = [clean_ids[i:i + batch_size] for i in range(0, len(clean_ids), batch_size)]
    dfs = []
    print(f"  • Querying Athena in {len(batches)} batch(es) for {len(clean_ids):,} IDs...")
    for idx, batch in enumerate(batches, 1):
        batch_ids_str = ", ".join(f"'{b}'" for b in batch)
        query = sql_template.format(batch_ids=batch_ids_str)
        b_df = wr.athena.read_sql_query(sql=query, database=ATHENA_DB, boto3_session=session, workgroup="metabase", s3_output=S3_OUTPUT, ctas_approach=False)
        dfs.append(b_df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# ==============================================================================
# STEP 1: EXTRACT ALL AT-LEAST-ONCE REACTIVATED LEADS (df11 -> reactivated_summary)
# ==============================================================================
print("\n[STEP 1/6] Extracting Reactivated Leads from Stage Histories...")
sql_step1 = """
    SELECT _id, lead, stage, iscurrent AS isCurrent, createdat AS createdAt, 
           reason, isstagemovement AS isStageMovement, comments 
    FROM "leadsdb-prod"."leadstagehistories" 
    WHERE (isstagemovement = 'true' OR isstagemovement = 'True' OR isstagemovement = '1')
"""
df11 = wr.athena.read_sql_query(sql=sql_step1, database=ATHENA_DB, boto3_session=session, workgroup="metabase", s3_output=S3_OUTPUT, ctas_approach=False)

df11['createdAt_dt'] = pd.to_datetime(df11['createdAt'], errors='coerce')
df_hist = df11.sort_values(by=['lead', 'createdAt_dt']).reset_index(drop=True)
df_hist['prev_stage'] = df_hist.groupby('lead')['stage'].shift(1)

mask_ni_transition = (df_hist['prev_stage'] == 'Not Interested') & (df_hist['stage'] != 'Not Interested')
mask_comment = df_hist['comments'].astype(str).str.lower().str.contains('reactivated', na=False)

reactivation_events = df_hist[mask_ni_transition | mask_comment].copy()
reactivated_summary = reactivation_events.groupby('lead').agg(
    total_reactivations=('createdAt', 'count'),
    first_reactivation_date=('createdAt_dt', 'min'),
    latest_reactivation_date=('createdAt_dt', 'max'),
    all_reactivation_dates=('createdAt', list)
).reset_index()

reactivated_leads = reactivated_summary['lead'].tolist()
print(f"  ✅ Found {len(reactivated_leads):,} At-Least-Once Reactivated Leads.")

# ==============================================================================
# STEP 2: FETCH BASE LEAD DETAILS (leads table -> df2)
# ==============================================================================
print("\n[STEP 2/6] Querying 'leads' Table for Reactivated Leads...")
sql_step2 = """
    SELECT _id AS lead, bornsource AS bornSource, source, secondarysource AS secondarySource,
           project, createdat AS createdAt, updatedat AS updatedAt
    FROM "leadsdb-prod"."leads"
    WHERE _id IN ({batch_ids})
"""
df2 = run_batched_athena_query(sql_step2, reactivated_leads, 'lead')

# ==============================================================================
# STEP 3: WEB VISITOR LOGS & PAGE TIMESPENTS (df3, df4, df6 -> df7)
# ==============================================================================
print("\n[STEP 3/6] Querying Web Visitor Logs & Aggregating Page Timespents...")
# A. globalvisitorlogs (df3)
sql_step3a = "SELECT _id, globalid AS globalId, visitorid AS visitorId FROM \"leadsdb-prod\".\"globalvisitorlogs\" WHERE globalid IN ({batch_ids})"
df3 = run_batched_athena_query(sql_step3a, df2['lead'].tolist(), 'globalId')

# B. visitoractivities (df6) -> df7 Page Timespent Pivots
sql_step3b = "SELECT _id, globalid AS globalId, pagename AS pageName, project, timespent AS timeSpent FROM \"leadsdb-prod\".\"visitoractivities\" WHERE globalid IN ({batch_ids})"
df6 = run_batched_athena_query(sql_step3b, df3['globalId'].tolist(), 'globalId')

target_pages = ['HOME', 'PRICE', 'PLAN', 'LOCATION', 'AMENITIES', 'MEDIA', 'SPECIFICATIONS']
df_temp = df6.copy()
df_temp['pageName_clean'] = df_temp['pageName'].astype(str).str.upper().str.strip()
df_temp['timeSpent'] = pd.to_numeric(df_temp['timeSpent'], errors='coerce').fillna(0.0)

df_filtered = df_temp[df_temp['pageName_clean'].isin(target_pages)]
df7 = df_filtered.pivot_table(index=['globalId', 'project'], columns='pageName_clean', values='timeSpent', aggfunc='sum', fill_value=0.0).reset_index()
df7.columns.name = None

for page in target_pages:
    ts_col = f"{page.lower()}_timespent"
    is_col = f"is_{page.lower()}"
    df7[ts_col] = df7[page].fillna(0.0) if page in df7.columns else 0.0
    if page in df7.columns: df7.drop(columns=[page], inplace=True)
    df7[is_col] = (df7[ts_col] > 0).astype(int)

# Total web timespent & frequency
df7['total_web_timespent'] = df7[[f"{p.lower()}_timespent" for p in target_pages]].sum(axis=1)

# ==============================================================================
# STEP 4: ACTION CLICK TRACKING (userredirecttracks -> df10)
# ==============================================================================
print("\n[STEP 4/6] Querying Action Clicks & Event Logs...")
sql_step4 = "SELECT _id, botlog AS botLog, projectid AS projectId, trackevents AS trackEvents FROM \"leadsdb-prod\".\"userredirecttracks\" WHERE botlog IN ({batch_ids})"
df9 = run_batched_athena_query(sql_step4, df3['visitorId'].tolist(), 'botLog')

def extract_event_flags(val):
    flags = {'is_site_visit_book_clicked': False, 'is_whatsapp_clicked': False, 'is_brochure_downloaded': False, 'is_price_sheet_viewed': False, 'is_cost_sheet_clicked': False}
    if isinstance(val, str) and val.strip():
        try: val = json.loads(val)
        except Exception: val = []
    if not isinstance(val, list): return flags
    for item in val:
        if isinstance(item, dict) and 'eventName' in item:
            evs = [item['eventName']] if isinstance(item['eventName'], str) else item['eventName']
            for ev in evs:
                if isinstance(ev, str):
                    u = ev.upper()
                    if any(k in u for k in ['BOOK_A_SITE_VISIT', 'SITE_VISIT']): flags['is_site_visit_book_clicked'] = True
                    if any(k in u for k in ['WHATSAPP', 'WA_']): flags['is_whatsapp_clicked'] = True
                    if any(k in u for k in ['BROCHURE', 'DOWNLOAD', 'PDF']): flags['is_brochure_downloaded'] = True
                    if any(k in u for k in ['PRICE', 'PRICESHEET']): flags['is_price_sheet_viewed'] = True
                    if any(k in u for k in ['COST_SHEET', 'COST']): flags['is_cost_sheet_clicked'] = True
    return flags

session_flags = pd.DataFrame(df9['trackEvents'].apply(extract_event_flags).tolist())
df10 = pd.concat([df9[['botLog']], session_flags], axis=1).groupby('botLog').max().reset_index()

# ==============================================================================
# STEP 5: TELECALL LOGS & WHATSAPP STATS (call, wa_stats)
# ==============================================================================
print("\n[STEP 5/6] Querying Telecall Records & WhatsApp Inbound Stats...")
# A. Call Records
sql_call = "SELECT _id, lead, createdat AS createdAt, type, status, duration FROM \"leadsdb-prod\".\"leadcallrecords\" WHERE lead IN ({batch_ids})"
call_df = run_batched_athena_query(sql_call, reactivated_leads, 'lead')
call_df['duration'] = pd.to_numeric(call_df['duration'], errors='coerce').fillna(0.0)

call_stats = call_df.groupby('lead').agg(
    total_attempted_calls=('type', 'count'),
    total_answered_calls=('status', lambda s: (s.astype(str).str.lower() == 'answered').sum()),
    total_outbound_call_duration=('duration', 'sum')
).reset_index()
call_stats['pickup_ratio'] = (call_stats['total_answered_calls'] / (call_stats['total_attempted_calls'] + 1e-5)).clip(0, 1)

# B. WhatsApp Inbound
sql_wa = "SELECT lead, time FROM \"leadsdb-prod\".\"whatsappchatmessages\" WHERE (isuser = 'true' OR isuser = 'True' OR isuser = '1') AND lead IN ({batch_ids})"
wa_df = run_batched_athena_query(sql_wa, reactivated_leads, 'lead')
wa_df['time_dt'] = pd.to_datetime(wa_df['time'], errors='coerce')

now_dt = pd.Timestamp.now()
wa_stats = wa_df.groupby('lead').agg(
    total_whatsapp_inbound=('time_dt', 'size'),
    days_since_whatsapp_inbound=('time_dt', lambda t: (now_dt - t.max()).total_seconds() / 86400.0)
).reset_index()

# ==============================================================================
# STEP 6: CONSOLIDATE MASTER TABLE & RUN ENSEMBLE MODEL
# ==============================================================================
print("\n[STEP 6/6] Consolidating Master Feature Matrix & Training Ensemble Model...")
master_df = reactivated_summary.merge(df2, on='lead', how='left')
master_df = master_df.merge(call_stats, on='lead', how='left')
master_df = master_df.merge(wa_stats, on='lead', how='left')

# Target Calculation
master_df['days_to_reactivation'] = (pd.to_datetime(master_df['latest_reactivation_date']) - pd.to_datetime(master_df['createdAt'])).dt.total_seconds() / 86400.0
master_df['reactivated_under_30_days'] = ((master_df['days_to_reactivation'] >= 0) & (master_df['days_to_reactivation'] < 30)).astype(int)

# Imputations
num_zeros = ['total_attempted_calls', 'total_answered_calls', 'pickup_ratio', 'total_outbound_call_duration', 'total_whatsapp_inbound']
for c in num_zeros: master_df[c] = master_df[c].fillna(0.0)
master_df['days_since_whatsapp_inbound'] = master_df['days_since_whatsapp_inbound'].fillna(3000.0)

# Feature Exclusion & Train/Test Split
exclude = ['lead', 'stage', 'createdAt', 'createdAt_dt', 'first_reactivation_date', 'latest_reactivation_date', 'all_reactivation_dates', 'reactivated_under_30_days', 'days_to_reactivation', 'bornSource', 'source', 'secondarySource', 'project', 'updatedAt']
X = master_df[[c for c in master_df.columns if c not in exclude]].select_dtypes(include=[np.number, bool]).copy()
y = master_df['reactivated_under_30_days']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

# Models
rf = RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_train, y_train)
ratio = (len(y_train) - y_train.sum()) / (y_train.sum() + 1e-5)
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.03, scale_pos_weight=ratio, random_state=42, n_jobs=-1).fit(X_train, y_train)
lgb = LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=31, is_unbalance=True, random_state=42, verbose=-1, n_jobs=-1).fit(X_train, y_train)

# Weighted Ensemble Prediction
p_ens = 0.30 * rf.predict_proba(X_test)[:, 1] + 0.35 * xgb.predict_proba(X_test)[:, 1] + 0.35 * lgb.predict_proba(X_test)[:, 1]
y_pred_ens = (p_ens >= 0.50).astype(int)

print("=" * 80)
print(f"🎉 PIPELINE COMPLETE! Ensemble ROC-AUC: {roc_auc_score(y_test, p_ens):.4f}")
print("=" * 80)
print(classification_report(y_test, y_pred_ens, target_names=["Wait (>30d)", "Priority (<30d)"]))


In [628]:
import awswrangler as wr
import boto3
import pandas as pd

session = boto3.Session(region_name="ap-south-1", profile_name='ASBL')

# 1. Compute 6 months ago cutoff date string
date_cutoff = (pd.Timestamp.now() - pd.DateOffset(months=6)).strftime("%Y-%m-%d %H:%M:%S")

print(f"Querying Athena for all leads created in the last 6 months (since {date_cutoff})...")

# 2. Athena query on leads table with date filter
sql_query = f"""
    SELECT 
        _id AS lead,
        bornsource AS bornSource,
        source,
        secondarysource AS secondarySource,
        project,
        createdat AS createdAt,
        updatedat AS updatedAt
    FROM "leadsdb-prod"."leads" 
    WHERE createdat >= TIMESTAMP '{date_cutoff}'
"""

_leadsDF_6m = wr.athena.read_sql_query(
    sql=sql_query,
    database="leadsdb-prod",
    boto3_session=session,
    workgroup="metabase",
    s3_output="s3://metabase-asbl/athena-query-results/",
    ctas_approach=False,
)

print("=" * 70)
print(f"🎉 SUCCESS! Loaded {_leadsDF_6m.shape[0]:,} total leads created in the last 6 months into `_leadsDF_6m`")
print("=" * 70)
print("DataFrame Columns:", _leadsDF_6m.columns.tolist())
_leadsDF_6m.head()


Querying Athena for all leads created in the last 6 months (since 2026-02-10 09:27:19)...
🎉 SUCCESS! Loaded 230,620 total leads created in the last 6 months into `_leadsDF_6m`
DataFrame Columns: ['lead', 'bornSource', 'source', 'secondarySource', 'project', 'createdAt', 'updatedAt']


,lead,bornSource,source,secondarySource,project,createdAt,updatedAt
0,6a2fa015c35b349ad1875764,69d4979d60cfeeb29bcd904d,69d4979d60cfeeb29bcd904d,[],68f228762292e21cc6165c2e,2026-06-15 06:47:49.225,2026-06-21 10:07:06.852
1,6a2fa015c35b349ad18757ad,69d4979d60cfeeb29bcd904d,69d4979d60cfeeb29bcd904d,[],68f228762292e21cc6165c2e,2026-06-15 06:47:49.689,2026-06-22 07:23:46.828
2,6a2fa016c35b349ad18757f6,69d4979d60cfeeb29bcd904d,69d4979d60cfeeb29bcd904d,[],68f228762292e21cc6165c2e,2026-06-15 06:47:50.189,2026-06-18 10:47:56.819
3,6a2fa016c35b349ad187583f,69d4979d60cfeeb29bcd904d,69d4979d60cfeeb29bcd904d,[],68f228762292e21cc6165c2e,2026-06-15 06:47:50.638,2026-06-16 06:54:12.244
4,6a2fa017c35b349ad1875888,69d4979d60cfeeb29bcd904d,69d4979d60cfeeb29bcd904d,[],68f228762292e21cc6165c2e,2026-06-15 06:47:51.080,2026-06-25 08:48:25.154


In [631]:
import awswrangler as wr
import boto3
import pandas as pd

# Extract unique lead list from _leadsDF_6m
lead_list_6m = _leadsDF_6m['lead'].dropna().astype(str).str.replace(r'\.0$', '', regex=True).unique().tolist()

batch_size = 500
lead_batches = [lead_list_6m[i:i + batch_size] for i in range(0, len(lead_list_6m), batch_size)]

session = boto3.Session(region_name="ap-south-1", profile_name='ASBL')
df_list = []

print(f"Executing Athena queries across {len(lead_batches)} batch(es) for {len(lead_list_6m):,} leads...")

for idx, batch in enumerate(lead_batches, 1):
    lead_ids_str = ", ".join(f"'{l}'" for l in batch)
    
    sql_query = f"""
        SELECT 
            _id,
            lead, 
            stage, 
            iscurrent AS isCurrent, 
            createdat AS createdAt, 
            reason, 
            isstagemovement AS isStageMovement, 
            cancelledreason AS cancellation_reason,
            comments 
        FROM "leadsdb-prod"."leadstagehistories" 
        WHERE lead IN ({lead_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False,
    )
    df_list.append(batch_df)
    if idx % 50 == 0 or idx == len(lead_batches):
        print(f"  • Batch {idx}/{len(lead_batches)} complete...")

_leadsStageDF_6m = pd.concat(df_list, ignore_index=True)
print(f"🎉 Loaded {_leadsStageDF_6m.shape[0]:,} rows into `_leadsStageDF_6m`.")


Executing Athena queries across 462 batch(es) for 230,620 leads...
  • Batch 50/462 complete...
  • Batch 100/462 complete...
  • Batch 150/462 complete...
  • Batch 200/462 complete...
  • Batch 250/462 complete...
  • Batch 300/462 complete...
  • Batch 350/462 complete...
  • Batch 400/462 complete...
  • Batch 450/462 complete...
  • Batch 462/462 complete...
🎉 Loaded 2,100,520 rows into `_leadsStageDF_6m`.


In [632]:
import time
import pandas as pd

print("🚀 Starting Episode Breakdown on _leadsStageDF_6m...")
t0 = time.time()

# 1. Ensure datetime format and sort chronologically by lead and timestamp
df_hist = _leadsStageDF_6m.copy()
df_hist['createdAt_dt'] = pd.to_datetime(df_hist['createdAt'], errors='coerce')
df_hist['lead_clean'] = df_hist['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
df_hist = df_hist.dropna(subset=['lead_clean', 'createdAt_dt']).sort_values(by=['lead_clean', 'createdAt_dt']).reset_index(drop=True)

# 2. Identify Episode Boundary (Stage is 'Not Interested' OR Last Record for that Lead)
df_hist['is_ni'] = df_hist['stage'].astype(str).str.strip() == 'Not Interested'
df_hist['is_last_lead_rec'] = df_hist['lead_clean'] != df_hist['lead_clean'].shift(-1)
df_hist['is_ep_end'] = df_hist['is_ni'] | df_hist['is_last_lead_rec']

# 3. Assign Episode Number per Lead via Cumulative Sum
df_hist['ep_end_shift'] = df_hist.groupby('lead_clean')['is_ep_end'].shift(1).fillna(False).astype(int)
df_hist['episode_number'] = df_hist.groupby('lead_clean')['ep_end_shift'].cumsum() + 1

# 4. Group by (lead, episode_number) to extract episode boundaries & timestamps
df_episodes = df_hist.groupby(['lead_clean', 'episode_number']).agg(
    episode_start_time=('createdAt_dt', 'min'),
    episode_end_time=('createdAt_dt', 'max'),
    start_stage=('stage', 'first'),
    end_stage=('stage', 'last'),
    total_stage_movements=('createdAt_dt', 'count')
).reset_index().rename(columns={'lead_clean': 'lead'})

# 5. Generate Episode ID format (x_y: leadId_episodeNumber)
df_episodes['episode_id'] = df_episodes['lead'] + '_' + df_episodes['episode_number'].astype(str)

# 6. Flag `is_reactivated_episode`: 1 if episode (x_y+1) exists for that lead, else 0
max_ep_per_lead = df_episodes.groupby('lead')['episode_number'].transform('max')
df_episodes['is_reactivated_episode'] = (df_episodes['episode_number'] < max_ep_per_lead).astype(int)

t1 = time.time()

print("=" * 80)
print(f"🎉 SUCCESS! Processed {len(_leadsStageDF_6m):,} stage rows into {len(df_episodes):,} Episodes in {t1 - t0:.2f} seconds!")
print("=" * 80)
print(f"  • Reactivated Episodes (is_reactivated_episode = 1): {df_episodes['is_reactivated_episode'].sum():,} ({(df_episodes['is_reactivated_episode'].sum()/len(df_episodes))*100:.2f}%)")
print(f"  • Latest / Non-Reactivated Episodes (0):            {len(df_episodes) - df_episodes['is_reactivated_episode'].sum():,}")
print("=" * 80)
df_episodes.head(12)


🚀 Starting Episode Breakdown on _leadsStageDF_6m...
🎉 SUCCESS! Processed 2,100,520 stage rows into 332,590 Episodes in 1.66 seconds!
  • Reactivated Episodes (is_reactivated_episode = 1): 102,006 (30.67%)
  • Latest / Non-Reactivated Episodes (0):            230,584


,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode
0,698afa2394d97ec3de58fb33,1,2026-02-10 09:28:04.260,2026-02-10 13:53:01.164,New Lead,Not Interested,5,698afa2394d97ec3de58fb33_1,0
1,698afa862810af121af4ccbb,1,2026-02-10 09:29:43.174,2026-02-10 13:33:42.030,New Lead,Not Interested,6,698afa862810af121af4ccbb_1,1
2,698afa862810af121af4ccbb,2,2026-02-10 13:35:35.874,2026-02-11 05:59:48.320,Lead Initiated,Not Interested,3,698afa862810af121af4ccbb_2,1
3,698afa862810af121af4ccbb,3,2026-02-19 14:07:58.022,2026-02-20 08:56:03.216,Lead Initiated,Not Interested,5,698afa862810af121af4ccbb_3,0
4,698afac035b8e1631ecf09ac,1,2026-02-10 09:30:40.449,2026-02-18 08:00:52.760,New Lead,Not Interested,12,698afac035b8e1631ecf09ac_1,1
5,698afac035b8e1631ecf09ac,2,2026-06-06 10:49:26.683,2026-06-06 10:49:26.683,Not Interested,Not Interested,1,698afac035b8e1631ecf09ac_2,0
6,698afadf2810af121af4ce21,1,2026-02-10 09:31:11.562,2026-02-14 08:05:32.472,New Lead,Not Interested,14,698afadf2810af121af4ce21_1,0
7,698afb172810af121af4d18f,1,2026-02-10 09:32:07.862,2026-02-10 09:48:03.040,New Lead,Not Interested,2,698afb172810af121af4d18f_1,0
8,698afb9079dc8582f2557aed,1,2026-02-10 09:34:08.821,2026-02-17 11:02:25.138,New Lead,Not Interested,13,698afb9079dc8582f2557aed_1,1
9,698afb9079dc8582f2557aed,2,2026-03-03 16:49:38.411,2026-03-03 16:49:38.411,Not Interested,Not Interested,1,698afb9079dc8582f2557aed_2,0


In [ ]:
import time
import pandas as pd


t0 = time.time()

df_hist = _leadsStageDF_6m.copy()
df_hist['createdAt_dt'] = pd.to_datetime(df_hist['createdAt'], errors='coerce')
df_hist['lead_clean'] = df_hist['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
df_hist = df_hist.dropna(subset=['lead_clean', 'createdAt_dt']).sort_values(by=['lead_clean', 'createdAt_dt']).reset_index(drop=True)


df_hist['prev_stage'] = df_hist.groupby('lead_clean')['stage'].shift(1)


is_duplicate_ni = (df_hist['stage'] == 'Not Interested') & (df_hist['prev_stage'] == 'Not Interested')
df_valid = df_hist[~is_duplicate_ni].copy().reset_index(drop=True)


df_valid['prev_stage'] = df_valid.groupby('lead_clean')['stage'].shift(1)


df_valid['is_first_ni'] = (df_valid['stage'] == 'Not Interested') & (df_valid['prev_stage'] != 'Not Interested')
df_valid['is_last_rec'] = df_valid['lead_clean'] != df_valid['lead_clean'].shift(-1)
df_valid['is_ep_end'] = df_valid['is_first_ni'] | df_valid['is_last_rec']


df_valid['ep_end_shift'] = df_valid.groupby('lead_clean')['is_ep_end'].shift(1).fillna(False).astype(int)
df_valid['episode_number'] = df_valid.groupby('lead_clean')['ep_end_shift'].cumsum() + 1


df_episodes = df_valid.groupby(['lead_clean', 'episode_number']).agg(
    episode_start_time=('createdAt_dt', 'min'),
    episode_end_time=('createdAt_dt', 'max'),
    start_stage=('stage', 'first'),
    end_stage=('stage', 'last'),
    total_stage_movements=('createdAt_dt', 'count')
).reset_index().rename(columns={'lead_clean': 'lead'})


df_episodes['episode_id'] = df_episodes['lead'] + '_' + df_episodes['episode_number'].astype(str)


max_ep_per_lead = df_episodes.groupby('lead')['episode_number'].transform('max')
df_episodes['is_reactivated_episode'] = (df_episodes['episode_number'] < max_ep_per_lead).astype(int)

t1 = time.time()


df_episodes.head(10)


🚀 Storing Episode Timestamps & Boundaries...
🎉 EPISODES STORED SUCCESSFULLY IN 1.53 SECONDS!
  • Total Clean Episodes:                    249,975
  • Reactivated Episodes (is_reactivated = 1): 19,391 (7.76%)
  • Latest / Ongoing Episodes (0):            230,584


,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode
0,698afa2394d97ec3de58fb33,1,2026-02-10 09:28:04.260,2026-02-10 13:53:01.164,New Lead,Not Interested,5,698afa2394d97ec3de58fb33_1,0
1,698afa862810af121af4ccbb,1,2026-02-10 09:29:43.174,2026-02-10 13:33:42.030,New Lead,Not Interested,6,698afa862810af121af4ccbb_1,1
2,698afa862810af121af4ccbb,2,2026-02-10 13:35:35.874,2026-02-11 05:59:48.320,Lead Initiated,Not Interested,3,698afa862810af121af4ccbb_2,1
3,698afa862810af121af4ccbb,3,2026-02-19 14:07:58.022,2026-02-20 08:56:03.216,Lead Initiated,Not Interested,5,698afa862810af121af4ccbb_3,0
4,698afac035b8e1631ecf09ac,1,2026-02-10 09:30:40.449,2026-02-18 08:00:52.760,New Lead,Not Interested,12,698afac035b8e1631ecf09ac_1,0
5,698afadf2810af121af4ce21,1,2026-02-10 09:31:11.562,2026-02-14 08:05:32.472,New Lead,Not Interested,14,698afadf2810af121af4ce21_1,0
6,698afb172810af121af4d18f,1,2026-02-10 09:32:07.862,2026-02-10 09:48:03.040,New Lead,Not Interested,2,698afb172810af121af4d18f_1,0
7,698afb9079dc8582f2557aed,1,2026-02-10 09:34:08.821,2026-02-17 11:02:25.138,New Lead,Not Interested,13,698afb9079dc8582f2557aed_1,0
8,698afc7435b8e1631ecf146a,1,2026-02-10 09:37:56.692,2026-02-10 10:27:04.499,New Lead,Not Interested,4,698afc7435b8e1631ecf146a_1,0
9,698afc8f94d97ec3de5901f0,1,2026-02-10 09:38:24.314,2026-02-12 07:28:48.800,New Lead,Not Interested,5,698afc8f94d97ec3de5901f0_1,0


In [814]:
df_episodes.columns

Index(['lead', 'episode_number', 'episode_start_time', 'episode_end_time',
       'start_stage', 'end_stage', 'total_stage_movements', 'episode_id',
       'is_reactivated_episode', 'lead_clean'],
      dtype='str')

In [642]:
import awswrangler as wr
import boto3
import pandas as pd

# 1. Clean unique lead IDs list directly from df_episodes['lead']
episode_lead_list = (
    df_episodes['lead']
    .dropna()
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
    .unique()
    .tolist()
)

# 2. Batching Setup (500 leads per batch to avoid Athena limits)
batch_size = 500
lead_batches = [episode_lead_list[i:i + batch_size] for i in range(0, len(episode_lead_list), batch_size)]

session = boto3.Session(region_name="ap-south-1", profile_name='ASBL')
call_list = []

print(f"Executing Athena query on 'leadcallrecords' across {len(lead_batches)} batch(es) for {len(episode_lead_list):,} episode leads...")

for idx, batch in enumerate(lead_batches, 1):
    batch_ids_str = ", ".join(f"'{l}'" for l in batch)
    
    # Direct Athena query using lead IN (df_episodes['lead'])
    sql_query = f"""
        SELECT 
            _id,
            lead,
            createdat AS createdAt,
            direction,
            callstatus AS status,
            callduration AS duration
        FROM "leadsdb-prod"."leadcallrecords" 
        WHERE lead IN ({batch_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False,
    )
    call_list.append(batch_df)
    if idx % 50 == 0 or idx == len(lead_batches):
        print(f"  • Batch {idx}/{len(lead_batches)} complete ({sum(len(d) for d in call_list):,} rows fetched)...")

# 3. Combine all batches into _callsDF_6m
_callsDF_6m = pd.concat(call_list, ignore_index=True)

print("=" * 70)
print(f"🎉 SUCCESS! Loaded {_callsDF_6m.shape[0]:,} total call records for `df_episodes` leads into `_callsDF_6m`")
print("=" * 70)
print("DataFrame Columns:", _callsDF_6m.columns.tolist())
_callsDF_6m.head()


Executing Athena query on 'leadcallrecords' across 462 batch(es) for 230,584 episode leads...
  • Batch 50/462 complete (126,791 rows fetched)...
  • Batch 100/462 complete (263,493 rows fetched)...
  • Batch 150/462 complete (412,665 rows fetched)...
  • Batch 200/462 complete (612,857 rows fetched)...
  • Batch 250/462 complete (788,452 rows fetched)...
  • Batch 300/462 complete (954,157 rows fetched)...
  • Batch 350/462 complete (1,025,348 rows fetched)...
  • Batch 400/462 complete (1,102,423 rows fetched)...
  • Batch 450/462 complete (1,232,324 rows fetched)...
  • Batch 462/462 complete (1,248,545 rows fetched)...
🎉 SUCCESS! Loaded 1,248,545 total call records for `df_episodes` leads into `_callsDF_6m`
DataFrame Columns: ['_id', 'lead', 'createdAt', 'direction', 'status', 'duration']


,_id,lead,createdAt,direction,status,duration
0,6a69cce6da35ce45f75dd5c8,698b78c755b4046b25abc5c0,2026-07-29 09:50:30.001,outbound,agent-missed,0
1,6a6b21a734ca599fc4987601,698b78c755b4046b25abc5c0,2026-07-30 10:04:23.708,outbound,customer-missed,0
2,6a6c8050dc3e1d4ead933a75,698b78c755b4046b25abc5c0,2026-07-31 11:00:32.937,outbound,answered,33.677
3,6a6d7c7797c85cc3c36f3aba,698b78c755b4046b25abc5c0,2026-08-01 04:56:23.836,outbound,agent-missed,0
4,6a6d91c0cdf751c9fbd6f332,698b78c755b4046b25abc5c0,2026-08-01 06:27:12.859,outbound,answered,12.814


In [647]:
import awswrangler as wr
import boto3
import pandas as pd

# 1. Clean unique lead IDs list directly from df_episodes['lead']
episode_lead_list = (
    df_episodes['lead']
    .dropna()
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
    .unique()
    .tolist()
)

# 2. Batching Setup (500 leads per batch to avoid Athena limits)
batch_size = 500
lead_batches = [episode_lead_list[i:i + batch_size] for i in range(0, len(episode_lead_list), batch_size)]

session = boto3.Session(region_name="ap-south-1", profile_name='ASBL')
call_list = []

print(f"Executing Athena query on 'leadcallrecords' across {len(lead_batches)} batch(es) for {len(episode_lead_list):,} episode leads...")

for idx, batch in enumerate(lead_batches, 1):
    batch_ids_str = ", ".join(f"'{l}'" for l in batch)
    
    # Direct Athena query using lead IN (df_episodes['lead'])
    sql_query = f"""
        SELECT *
        FROM "leadsdb-prod"."leads" 
        WHERE _id IN ({batch_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False,
    )
    call_list.append(batch_df)
    if idx % 50 == 0 or idx == len(lead_batches):
        print(f"  • Batch {idx}/{len(lead_batches)} complete ({sum(len(d) for d in call_list):,} rows fetched)...")

# 3. Combine all batches into _callsDF_6m
_leadsDF_6m = pd.concat(call_list, ignore_index=True)

print("=" * 70)
print(f"🎉 SUCCESS! Loaded {_leadsDF_6m.shape[0]:,} total call records for `df_episodes` leads into `_callsDF_6m`")
print("=" * 70)
print("DataFrame Columns:", _leadsDF_6m.columns.tolist())
_leadsDF_6m.head()


Executing Athena query on 'leadcallrecords' across 462 batch(es) for 230,584 episode leads...
  • Batch 50/462 complete (25,000 rows fetched)...
  • Batch 100/462 complete (50,000 rows fetched)...
  • Batch 150/462 complete (75,000 rows fetched)...
  • Batch 200/462 complete (100,000 rows fetched)...
  • Batch 250/462 complete (125,000 rows fetched)...
  • Batch 300/462 complete (150,000 rows fetched)...
  • Batch 350/462 complete (175,000 rows fetched)...
  • Batch 400/462 complete (200,000 rows fetched)...
  • Batch 450/462 complete (225,000 rows fetched)...
  • Batch 462/462 complete (230,584 rows fetched)...
🎉 SUCCESS! Loaded 230,584 total call records for `df_episodes` leads into `_callsDF_6m`
DataFrame Columns: ['_id', '__v', 'active', 'agreedtnc', 'alternatecontactnumber', 'assignedto', 'attendedevents', 'autotriggercalltime', 'bornsource', 'borntimeslot', 'botlogid', 'budget', 'callinprogress', 'callrequested', 'callingexecutive', 'cancelledreason', 'category', 'channelpartnere

,_id,__v,active,agreedtnc,alternatecontactnumber,assignedto,attendedevents,autotriggercalltime,bornsource,borntimeslot,botlogid,budget,callinprogress,callrequested,callingexecutive,cancelledreason,category,channelpartneremployee,channelpartnerref,city,comments,communicationemail,company,confirmed,contactnumber,country,createdat,createdby,creationtype,crmagent,crmflatreferrer,ctainactivitymessagesentat,date,emailid,employment,executivereferal,flatintrestedsofar,flatnumber,flatsclosed,fullname,globalid,goodlead,goodleadqualifiedat,googleadsformname,homes,isaurumdetailsfetched,iscronlead,ispurplelead,isscriptupdate,issitevisitbooked,iswatchlist,lastactivity,lastcostsheetupdated,lastname,laststageupdated,laststatusupdated,lastupdateddate,latestattendedwebinar,latestcategoryresponse,latestreferral,latestregisteredwebinar,leadcreationsource,leadgenids,leadid,leadintent,leadreferral,leadsourcemode,linkedinprofileurl,location,loggedby,managercomments,maritalstatus,masterleadid,maxtransitionedstagehistory,micromarket,mygate,needsconfirmation,onlinestatus,onlinestatus:isonline,optid,optintime,optedin,orangeleadtime,preferredautocallchannel,priceagreedsofar,priceclosed,project,purpleleadtime,purpose,qualifiedat,qualifiednewlead,questionnaire,questions,rawcontactnumber,reactivatedat,reffrom,referral,referralcomments,registeredevents,respondedtime,salary,salutation,secondarysource,secondaryvisitors,sendtoconversationfrombulk,sessionbotlogid,sfcalldate,showresponsetime,sitevisitbookingtime,sitevisitduration,sitevisitexecutive,source,sourcehistory,stage,stagehistory,state,status,statuscreation,statusfilterapplydata,statushistory,statusstagemapper,timeline,timezone,totalduration,upcomingcalldetails,updatedat,utm_campaign,viewed,visitorid,voizformresponse,watchlistedat,whatsappcontactnumber,zipcodeaddress,zoomlead,createdat_ist,date_ist,updatedat_ist
0,698afa2394d97ec3de58fb33,0,true,<NA>,<NA>,674817a7986888f2ced7773b,[],NaT,66683e8765601d05fa511df3,12:00PM - 03:00PM,<NA>,<NA>,false,<NA>,674817a7986888f2ced7773b,<NA>,<NA>,<NA>,<NA>,<NA>,He said not looking any flats for purchase,<NA>,<NA>,<NA>,+919xxxxxxxx,India,2026-02-10 09:28:03.907,<NA>,<NA>,<NA>,<NA>,NaT,NaT,g20260811:LFm+QdDfzz8ybn/ayhXpOlcz6XJ0rEewCNKt...,<NA>,<NA>,<NA>,<NA>,<NA>,g20260811:5s7kCResSPgjZek4wi3ja/b1eOd1+GS9d7b8...,<NA>,<NA>,NaT,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,false,2026-02-09 15:50:14.939,2026-02-10 13:52:17.911,<NA>,2026-02-10 13:53:01.194,2026-02-10 13:53:01.194,2026-02-10 13:53:01.455,<NA>,<NA>,<NA>,<NA>,<NA>,[],1052636,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,836421,698afbd0ee9d39041814f4a9,<NA>,"{""society"": null, ""city"": null, ""questionnaire...",<NA>,"{""totalWebTime"": 0}",<NA>,5726299479535566995,2026-06-22 14:48:40.496,true,NaT,<NA>,<NA>,<NA>,65adfbb3f09468f0e32fba49,NaT,<NA>,NaT,<NA>,<NA>,<NA>,g20260811:OCLUBYSJujJghmgWxQ6ZL1Cr+e2uWSyTwvFB...,NaT,<NA>,<NA>,<NA>,[],2026-02-10 09:34:45.403,<NA>,<NA>,[],<NA>,<NA>,<NA>,NaT,true,NaT,<NA>,<NA>,66683e8765601d05fa511df3,[],<NA>,698b383d5866d3e87ad61020,<NA>,<NA>,NaT,NaT,<NA>,<NA>,<NA>,<NA>,69,<NA>,2026-06-22 14:48:40.496,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,[],<NA>,2026-02-10 14:58:03.907,NaT,2026-06-22 20:18:40.496
1,698afa862810af121af4ccbb,0,true,<NA>,<NA>,693bb57b88ac9b7028b11f2e,[],NaT,66683e8865601d05fa511dfd,12:00PM - 03:00PM,<NA>,<NA>,false,<NA>,693bb57b88ac9b7028b11f2e,<NA>,<NA>,<NA>,<NA>,<NA>,Rural lead,<NA>,<NA>,<NA>,+919xxxxxxxx,India,2026-02-10 09:29:42.814,<NA>,<NA>,<NA>,<NA>,NaT,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,g20260811:Bq9UmWYpoPU+ee6ekZBMREyjlX+1jKVBFwDm...,<NA>,<NA>,NaT,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,false,2026-02-09 15:41:19.280,2026-02-20 08:55:20.256,<NA>,2026-02-20 08:56:03.239,2026-02-20 08:56:03.239,2026-02-20 08:56:03.424,<NA>,<NA>,<NA>,<NA>,<NA>,[],1052637,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,836422,698b04fe25611c5bc00bc68e,<NA>,<NA>,<NA>,"{""totalWebTime"": 0}",<NA>,5630468548632322267,2026-02-10 09:29:43.307,true,NaT,<NA>,<NA>,<NA>,65adfbb3f09468f0e32fba49,NaT,<NA>,NaT,<NA>,<NA>,<NA>,g202608

In [ ]:
import awswrangler as wr
import boto3
import pandas as pd

session = boto3.Session(region_name="ap-south-1", profile_name="ASBL")
ATHENA_DB = "leadsdb-prod"
S3_OUTPUT = "s3://metabase-asbl/athena-query-results/"

def run_batched_athena_query(sql_template, id_list, batch_size=500):
    
    clean_ids = list(set([str(x).replace('.0', '').strip() for x in id_list if pd.notna(x)]))
    batches = [clean_ids[i:i + batch_size] for i in range(0, len(clean_ids), batch_size)]
    dfs = []
    print(f"  • Querying Athena in {len(batches)} batch(es) for {len(clean_ids):,} IDs...")
    for idx, batch in enumerate(batches, 1):
        batch_ids_str = ", ".join(f"'{b}'" for b in batch)
        query = sql_template.format(batch_ids=batch_ids_str)
        b_df = wr.athena.read_sql_query(sql=query, database=ATHENA_DB, boto3_session=session, workgroup="metabase", s3_output=S3_OUTPUT, ctas_approach=False)
        dfs.append(b_df)
        if idx % 50 == 0 or idx == len(batches):
            print(f"    - Batch {idx}/{len(batches)} complete ({sum(len(d) for d in dfs):,} rows)...")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# ==============================================================================
# A. QUERY 'leads' TABLE -> df2 (Map lead -> globalId)
# ==============================================================================

# ==============================================================================
# B. QUERY 'globalvisitorlogs' TABLE -> df3 (Map globalId -> visitorId)
# ==============================================================================


# ==============================================================================
# C. QUERY 'visitoractivities' TABLE -> df6 (Page Views & Timespents)
# ==============================================================================
print("\n[C] Querying 'visitoractivities' table for page-level timespents...")
sql_df6 = """
    SELECT 
        _id, 
        globalid AS globalId, 
        pagename AS pageName, 
        project, 
        timespent AS timeSpent, 
        createdat AS createdAt 
    FROM "leadsdb-prod"."visitoractivities" 
    WHERE globalid IN ({batch_ids})
"""
df6 = run_batched_athena_query(sql_df6, df3['globalId'].tolist())
print(f"  ✅ df6 Loaded: {len(df6):,} page view activity records.")

# ==============================================================================
# D. QUERY 'visitorlogs' TABLE -> df4 (Web Sessions)
# ==============================================================================


# ==============================================================================
# E. QUERY 'userredirecttracks' TABLE -> df9 (Action Click Events)
# ==============================================================================
print("\n[E] Querying 'userredirecttracks' table for action click events...")
sql_df9 = """
    SELECT 
        
        botlog AS botLog, 
        trackevents AS trackEvents,  
    FROM "leadsdb-prod"."userredirecttracks" 
    WHERE botlog IN ({batch_ids})
"""
df9 = run_batched_athena_query(sql_df9, df4['_id'].tolist())
print(f"  ✅ df9 Loaded: {len(df9):,} user action redirect event records.")

print("=" * 80)
print("🎉 ALL WEBSITE DATA TABLES LOADED SUCCESSFULLY!")
print("=" * 80)


In [653]:
import pandas as pd

# 1. List of strictly relevant lead metadata columns to keep
relevant_cols = [
    '_id', 'masterleadid', 'bornsource', 'source', 'secondarysource', 
     'project',  'status', 
     'createdat',  
   'globalid', 'leadid'
]

# Keep only columns that exist in _leadsDF_6m
available_relevant_cols = [c for c in relevant_cols if c in _leadsDF_6m.columns]

# 2. Create clean copy of _leadsDF_6m with relevant columns only
_leadsDF_6m_clean = _leadsDF_6m[available_relevant_cols].copy()

# Rename columns to standard camelCase for clarity if needed
column_rename_map = {
    '_id': '_id',
    'leadid' : 'leadId',
    'masterleadid': 'masterLeadId',
    'bornsource': 'bornSource',
    'secondarysource': 'secondarySource',
    'createdat': 'lead_createdAt',
    'globalid': 'globalId'
}
_leadsDF_6m_clean.rename(columns={k: v for k, v in column_rename_map.items() if k in _leadsDF_6m_clean.columns}, inplace=True)

# 3. Clean merge keys for exact matching
df_episodes['lead_clean'] = df_episodes['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
_leadsDF_6m_clean['_id_clean'] = _leadsDF_6m_clean['_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# 4. Merge df_episodes with _leadsDF_6m_clean to create df1
df1 = pd.merge(
    df_episodes,
    _leadsDF_6m_clean.drop(columns=['_id'], errors='ignore'),
    left_on='lead_clean',
    right_on='_id_clean',
    how='left'
).drop(columns=['lead_clean', '_id_clean'], errors='ignore')
df1.head(10)



,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId
0,698afa2394d97ec3de58fb33,1,2026-02-10 09:28:04.260,2026-02-10 13:53:01.164,New Lead,Not Interested,5,698afa2394d97ec3de58fb33_1,0,836421,66683e8765601d05fa511df3,66683e8765601d05fa511df3,[],65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:28:03.907,<NA>,1052636
1,698afa862810af121af4ccbb,1,2026-02-10 09:29:43.174,2026-02-10 13:33:42.030,New Lead,Not Interested,6,698afa862810af121af4ccbb_1,1,836422,66683e8865601d05fa511dfd,66683e8865601d05fa511dfd,"[{""source"": ""66683e8865601d05fa511dfd"", ""date""...",65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:29:42.814,<NA>,1052637
2,698afa862810af121af4ccbb,2,2026-02-10 13:35:35.874,2026-02-11 05:59:48.320,Lead Initiated,Not Interested,3,698afa862810af121af4ccbb_2,1,836422,66683e8865601d05fa511dfd,66683e8865601d05fa511dfd,"[{""source"": ""66683e8865601d05fa511dfd"", ""date""...",65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:29:42.814,<NA>,1052637
3,698afa862810af121af4ccbb,3,2026-02-19 14:07:58.022,2026-02-20 08:56:03.216,Lead Initiated,Not Interested,5,698afa862810af121af4ccbb_3,0,836422,66683e8865601d05fa511dfd,66683e8865601d05fa511dfd,"[{""source"": ""66683e8865601d05fa511dfd"", ""date""...",65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:29:42.814,<NA>,1052637
4,698afac035b8e1631ecf09ac,1,2026-02-10 09:30:40.449,2026-02-18 08:00:52.760,New Lead,Not Interested,12,698afac035b8e1631ecf09ac_1,0,836423,653a12681b2c61a972707464,653a12681b2c61a972707464,[],64a5178ed9039b2988d6e4ab,<NA>,2026-02-10 09:30:40.087,<NA>,1052638
5,698afadf2810af121af4ce21,1,2026-02-10 09:31:11.562,2026-02-14 08:05:32.472,New Lead,Not Interested,14,698afadf2810af121af4ce21_1,0,629056,673b0ad0b768a4ad714460a1,673b0ad0b768a4ad714460a1,[],671353e397ece153c124590c,<NA>,2026-02-10 09:31:11.345,<NA>,1052639
6,698afb172810af121af4d18f,1,2026-02-10 09:32:07.862,2026-02-10 09:48:03.040,New Lead,Not Interested,2,698afb172810af121af4d18f_1,0,836424,653a12681b2c61a972707463,653a12681b2c61a972707463,[],60dd82c75aec32505d9af7d5,<NA>,2026-02-10 09:32:07.518,<NA>,1052640
7,698afb9079dc8582f2557aed,1,2026-02-10 09:34:08.821,2026-02-17 11:02:25.138,New Lead,Not Interested,13,698afb9079dc8582f2557aed_1,0,836425,66683e8765601d05fa511df3,66683e8765601d05fa511df3,[],65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:34:08.466,<NA>,1052641
8,698afc7435b8e1631ecf146a,1,2026-02-10 09:37:56.692,2026-02-10 10:27:04.499,New Lead,Not Interested,4,698afc7435b8e1631ecf146a_1,0,836426,64a7859b0fcd685ef70d6de1,64a7859b0fcd685ef70d6de1,"[{""source"": ""64a7859b0fcd685ef70d6de1"", ""date""...",60dd82c75aec32505d9af7d5,<NA>,2026-02-10 09:37:56.350,<NA>,1052642
9,698afc8f94d97ec3de5901f0,1,2026-02-10 09:38:24.314,2026-02-12 07:28:48.800,New Lead,Not Interested,5,698afc8f94d97ec3de5901f0_1,0,836427,653a12681b2c61a972707464,653a12681b2c61a972707464,[],64a5178ed9039b2988d6e4ab,<NA>,2026-02-10 09:38:23.957,<NA>,1052643


In [655]:
import awswrangler as wr
import boto3
import pandas as pd

# 1. Clean unique ID list from df1['globalId'] (with fallback to df1['lead'])
id_col = 'globalId' if ('globalId' in df1.columns and df1['globalId'].notnull().any()) else 'lead'
global_id_list = df1[id_col].dropna().astype(str).str.replace(r'\.0$', '', regex=True).str.strip().unique().tolist()

# 2. Batching Setup (500 IDs per batch)
batch_size = 500
global_batches = [global_id_list[i:i + batch_size] for i in range(0, len(global_id_list), batch_size)]

session = boto3.Session(region_name="ap-south-1", profile_name="ASBL")
df3_list = []

print(f"Executing Athena query on 'globalvisitorlogs' across {len(global_batches)} batch(es) for {len(global_id_list):,} unique globalIds...")

for idx, batch in enumerate(global_batches, 1):
    batch_ids_str = ", ".join(f"'{g}'" for g in batch)
    
    # Query globalvisitorlogs table
    sql_query = f"""
        SELECT 
            globalid AS globalId, 
            visitorid AS visitorId, 
            project
        FROM "leadsdb-prod"."globalvisitorlogs" 
        WHERE globalid IN ({batch_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False,
    )
    df3_list.append(batch_df)
    if idx % 50 == 0 or idx == len(global_batches):
        print(f"  • Batch {idx}/{len(global_batches)} complete ({sum(len(d) for d in df3_list):,} total rows)...")

# 3. Combine batches into df3
df3 = pd.concat(df3_list, ignore_index=True) if df3_list else pd.DataFrame(columns=['_id', 'globalId', 'visitorId', 'createdAt'])

print("=" * 75)
print(f"🎉 SUCCESS! Loaded {df3.shape[0]:,} total records into `df3` from 'globalvisitorlogs'")
print("=" * 75)
print("DataFrame Columns:", df3.columns.tolist())
df3.head()


Executing Athena query on 'globalvisitorlogs' across 101 batch(es) for 50,341 unique globalIds...
  • Batch 50/101 complete (39,205 total rows)...
  • Batch 100/101 complete (74,560 total rows)...
  • Batch 101/101 complete (74,999 total rows)...
🎉 SUCCESS! Loaded 74,999 total records into `df3` from 'globalvisitorlogs'
DataFrame Columns: ['globalId', 'visitorId', 'project']


,globalId,visitorId,project
0,2464341,Visitor-2543864,5f9ff649ffef1e546c79ce9d
1,2464341,Visitor-3703498,60dd82c75aec32505d9af7d5
2,2464341,Visitor-2543611,64a5178ed9039b2988d6e4ab
3,2464341,Visitor-3703469,667d2f5c2eb25893ac4e5c10
4,2471457,Visitor-2551662,60dab07889ac0911e6668dab


In [657]:
df4=df1.merge(df3, on=['project','globalId'], how='left')

In [661]:
df4

,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId,visitorId
0,698afa2394d97ec3de58fb33,1,2026-02-10 09:28:04.260,2026-02-10 13:53:01.164,New Lead,Not Interested,5,698afa2394d97ec3de58fb33_1,0,836421,66683e8765601d05fa511df3,66683e8765601d05fa511df3,[],65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:28:03.907,<NA>,1052636,<NA>
1,698afa862810af121af4ccbb,1,2026-02-10 09:29:43.174,2026-02-10 13:33:42.030,New Lead,Not Interested,6,698afa862810af121af4ccbb_1,1,836422,66683e8865601d05fa511dfd,66683e8865601d05fa511dfd,"[{""source"": ""66683e8865601d05fa511dfd"", ""date""...",65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:29:42.814,<NA>,1052637,<NA>
2,698afa862810af121af4ccbb,2,2026-02-10 13:35:35.874,2026-02-11 05:59:48.320,Lead Initiated,Not Interested,3,698afa862810af121af4ccbb_2,1,836422,66683e8865601d05fa511dfd,66683e8865601d05fa511dfd,"[{""source"": ""66683e8865601d05fa511dfd"", ""date""...",65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:29:42.814,<NA>,1052637,<NA>
3,698afa862810af121af4ccbb,3,2026-02-19 14:07:58.022,2026-02-20 08:56:03.216,Lead Initiated,Not Interested,5,698afa862810af121af4ccbb_3,0,836422,66683e8865601d05fa511dfd,66683e8865601d05fa511dfd,"[{""source"": ""66683e8865601d05fa511dfd"", ""date""...",65adfbb3f09468f0e32fba49,<NA>,2026-02-10 09:29:42.814,<NA>,1052637,<NA>
4,698afac035b8e1631ecf09ac,1,2026-02-10 09:30:40.449,2026-02-18 08:00:52.760,New Lead,Not Interested,12,698afac035b8e1631ecf09ac_1,0,836423,653a12681b2c61a972707464,653a12681b2c61a972707464,[],64a5178ed9039b2988d6e4ab,<NA>,2026-02-10 09:30:40.087,<NA>,1052638,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249970,6a792178a03a0a265480d9d7,1,2026-08-10 00:55:20.575,2026-08-10 00:55:20.575,New Lead,New Lead,1,6a792178a03a0a265480d9d7_1,0,1004566,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-08-10 00:55:20.244,23831789,1283426,Visitor-27976922
249971,6a7921fb7711d0c08082037f,1,2026-08-10 00:57:31.841,2026-08-10 00:57:31.841,New Lead,New Lead,1,6a7921fb7711d0c08082037f_1,0,1004567,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,"[{""source"": ""69d8ae93e2ee6bc0cbfe6674"", ""date""...",68f228762292e21cc6165c2e,<NA>,2026-08-10 00:57:31.492,<NA>,1283427,<NA>
249972,6a792237a03a0a265480df83,1,2026-08-10 00:58:31.357,2026-08-10 00:58:31.357,New Lead,New Lead,1,6a792237a03a0a265480df83_1,0,1004568,673b0ad0b768a4ad714460b3,673b0ad0b768a4ad714460b3,[],671353e397ece153c124590c,<NA>,2026-08-10 00:58:31.083,17573136,1283428,Visitor-20177790
249973,6a7923b3a03a0a265480e7c3,1,2026-08-10 01:04:52.145,2026-08-10 01:04:52.145,New Lead,New Lead,1,6a7923b3a03a0a265480e7c3_1,0,1004569,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-08-10 01:04:51.181,23831853,1283429,Visitor-27976988


In [669]:
import awswrangler as wr
import boto3
import pandas as pd

# 1. 3-Month Date Filter
three_months_ago = pd.Timestamp.now() - pd.DateOffset(months=3)
three_months_str = three_months_ago.strftime("%Y-%m-%d %H:%M:%S")

print(f"\nFiltering episodes for the LAST 3 MONTHS ONLY (since {three_months_str})...")

# 2. Filter df4 for leads/episodes created in the last 3 months -> df5
df5 = df4[pd.to_datetime(df4['lead_createdAt'], errors='coerce') >= three_months_ago].copy().reset_index(drop=True)
print(f"  🎉 SUCCESS! Created `df5` with {len(df5):,} rows in the last 3 months.")

# 3. Extract unique visitor IDs from df5
visitor_list_3m = df5['visitorId'].dropna().astype(str).str.replace(r'\.0$', '', regex=True).str.strip().unique().tolist()

batch_size = 500
visitor_batches = [visitor_list_3m[i:i + batch_size] for i in range(0, len(visitor_list_3m), batch_size)]
df4_list = []

print(f"\nQuerying 'visitorlogs' table for {len(visitor_list_3m):,} unique visitorIds across {len(visitor_batches)} batch(es)...")

for idx, batch in enumerate(visitor_batches, 1):
    batch_ids_str = ", ".join(f"'{v}'" for v in batch)
    
    # Corrected SQL Query with proper commas and WHERE clause
    sql_query = f"""
        SELECT 
            _id, 
            visitorid AS visitorId, 
            globalid AS globalId, 
            cummulativetimespent AS cummulativeTimeSpent,
            createdat AS createdAt,
            instancenumber AS instanceNumber,
            pages,
            pricesheetsopened AS priceSheetsOpened,
            whatsappclicked AS whatsappClicked
        FROM "leadsdb-prod"."visitorlogs" 
        WHERE createdat >= TIMESTAMP '{three_months_str}'
          AND visitorid IN ({batch_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False
    )
    df4_list.append(batch_df)
    if idx % 50 == 0 or idx == len(visitor_batches):
        print(f"  • Batch {idx}/{len(visitor_batches)} complete ({sum(len(d) for d in df4_list):,} total rows)...")

# 4. Combine all batches into df6
df6 = pd.concat(df4_list, ignore_index=True) if df4_list else pd.DataFrame(
    columns=['_id', 'visitorId', 'globalId', 'cummulativeTimeSpent', 'createdAt', 'instanceNumber', 'pages', 'priceSheetsOpened', 'whatsappClicked']
)

print("=" * 75)
print(f"🎉 SUCCESS! Loaded {df6.shape[0]:,} total session records into `df6` from 'visitorlogs' (Last 3 Months)")
print("=" * 75)
print("DataFrame Columns:", df6.columns.tolist())
df6.head()



Filtering episodes for the LAST 3 MONTHS ONLY (since 2026-05-11 20:11:09)...
  🎉 SUCCESS! Created `df5` with 111,738 rows in the last 3 months.

Querying 'visitorlogs' table for 18,896 unique visitorIds across 38 batch(es)...
  • Batch 38/38 complete (43,961 total rows)...
🎉 SUCCESS! Loaded 43,961 total session records into `df6` from 'visitorlogs' (Last 3 Months)
DataFrame Columns: ['_id', 'visitorId', 'globalId', 'cummulativeTimeSpent', 'createdAt', 'instanceNumber', 'pages', 'priceSheetsOpened', 'whatsappClicked']


,_id,visitorId,globalId,cummulativeTimeSpent,createdAt,instanceNumber,pages,priceSheetsOpened,whatsappClicked
0,6a631156babc5fbc77dd1cb9,Visitor-7013554,<NA>,0,2026-07-24 07:16:38.800,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>
1,6a631156f3fe2faf465e10c3,Visitor-7013554,<NA>,0,2026-07-24 07:16:38.801,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>
2,6a6311ad630bca1f2039f763,Visitor-7013554,<NA>,0,2026-07-24 07:18:05.080,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>
3,6a6311c11d625ab3f59c5a60,Visitor-7013554,<NA>,0,2026-07-24 07:18:25.190,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>
4,6a670fb1399e0baa8e4e0aab,Visitor-7013554,<NA>,0,2026-07-27 07:58:41.587,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>


In [676]:
import awswrangler as wr
import boto3
import pandas as pd

# 1. Clean unique botlog ID list from df6['_id']
botlog_list = (
    df6['_id']
    .dropna()
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
    .unique()
    .tolist()
)

# 2. Batching Setup (500 botlogs per batch to prevent Athena query length limits)
batch_size = 500
botlog_batches = [botlog_list[i:i + batch_size] for i in range(0, len(botlog_list), batch_size)]

session = boto3.Session(region_name="ap-south-1", profile_name="ASBL")
df7_list = []

print(f"Executing Athena query on 'userredirecttracks' for {len(botlog_list):,} botlog IDs across {len(botlog_batches)} batch(es)...")

for idx, batch in enumerate(botlog_batches, 1):
    batch_ids_str = ", ".join(f"'{b}'" for b in batch)
    
    # Query userredirecttracks table
    sql_query = f"""
        SELECT 
            botlog AS botLog, 
            trackevents AS trackEvents
        FROM "leadsdb-prod"."userredirecttracks" 
        WHERE botlog IN ({batch_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False,
    )
    df7_list.append(batch_df)
    if idx % 50 == 0 or idx == len(botlog_batches):
        print(f"  • Batch {idx}/{len(botlog_batches)} complete ({sum(len(d) for d in df7_list):,} total rows)...")

# 3. Combine batches into df7
df7 = pd.concat(df7_list, ignore_index=True) if df7_list else pd.DataFrame(
    columns=[ 'botLog',  'trackEvents' ]
)

print("=" * 80)
print(f"🎉 SUCCESS! Loaded {df7.shape[0]:,} total user action redirect records into `df7` from 'userredirecttracks'")
print("=" * 80)
print("DataFrame Columns:", df7.columns.tolist())
df7.head()


Executing Athena query on 'userredirecttracks' for 43,961 botlog IDs across 88 batch(es)...
  • Batch 50/88 complete (24,997 total rows)...
  • Batch 88/88 complete (43,954 total rows)...
🎉 SUCCESS! Loaded 43,954 total user action redirect records into `df7` from 'userredirecttracks'
DataFrame Columns: ['botLog', 'trackEvents']


,botLog,trackEvents
0,6a023a111cfdd50b4d639f80,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
1,6a023a119a93b65ed4c594f9,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
2,6a023c99ddaa0af3065f4efb,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
3,6a023ca8f0a0af2c8db8c713,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
4,6a023d10b1b85af895ce181a,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."


In [678]:
df6=df6.merge(df7, left_on='_id', right_on='botLog', how='left')

In [681]:
df5

,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId,visitorId
0,6a023a53d90c4857b766189c,1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,New Lead,Not Interested,15,6a023a53d90c4857b766189c_1,0,920554,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:21:39.461,22787547,1177568,Visitor-26920396
1,6a023b888b2d7b451592811f,1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,New Lead,Not Interested,2,6a023b888b2d7b451592811f_1,0,920555,66d6b121137a13b8864d16a2,66d6b121137a13b8864d16a2,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:26:48.757,<NA>,1177569,<NA>
2,6a023bb0f858b6e164691c23,1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,New Lead,Not Interested,9,6a023bb0f858b6e164691c23_1,0,920556,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:27:28.565,<NA>,1177570,<NA>
3,6a023c83ddaa0af3065f4e91,1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,New Lead,Not Interested,7,6a023c83ddaa0af3065f4e91_1,0,920557,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:30:59.331,<NA>,1177571,<NA>
4,6a023d25d90c4857b7661b5d,1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,New Lead,Not Interested,10,6a023d25d90c4857b7661b5d_1,0,801227,69d4979d60cfeeb29bcd905b,69d4979d60cfeeb29bcd905b,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:33:41.149,<NA>,1177572,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111733,6a792178a03a0a265480d9d7,1,2026-08-10 00:55:20.575,2026-08-10 00:55:20.575,New Lead,New Lead,1,6a792178a03a0a265480d9d7_1,0,1004566,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-08-10 00:55:20.244,23831789,1283426,Visitor-27976922
111734,6a7921fb7711d0c08082037f,1,2026-08-10 00:57:31.841,2026-08-10 00:57:31.841,New Lead,New Lead,1,6a7921fb7711d0c08082037f_1,0,1004567,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,"[{""source"": ""69d8ae93e2ee6bc0cbfe6674"", ""date""...",68f228762292e21cc6165c2e,<NA>,2026-08-10 00:57:31.492,<NA>,1283427,<NA>
111735,6a792237a03a0a265480df83,1,2026-08-10 00:58:31.357,2026-08-10 00:58:31.357,New Lead,New Lead,1,6a792237a03a0a265480df83_1,0,1004568,673b0ad0b768a4ad714460b3,673b0ad0b768a4ad714460b3,[],671353e397ece153c124590c,<NA>,2026-08-10 00:58:31.083,17573136,1283428,Visitor-20177790
111736,6a7923b3a03a0a265480e7c3,1,2026-08-10 01:04:52.145,2026-08-10 01:04:52.145,New Lead,New Lead,1,6a7923b3a03a0a265480e7c3_1,0,1004569,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-08-10 01:04:51.181,23831853,1283429,Visitor-27976988


In [682]:
import awswrangler as wr
import boto3
import pandas as pd

# 1. Clean unique lead IDs list from df1['lead']
lead_list_wa = (
    df5['lead']
    .dropna()
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
    .unique()
    .tolist()
)

# 2. Batching Setup (500 leads per batch to prevent Athena SQL string limit errors)
batch_size = 500
lead_batches = [lead_list_wa[i:i + batch_size] for i in range(0, len(lead_list_wa), batch_size)]

session = boto3.Session(region_name="ap-south-1", profile_name="ASBL")
wa_parts = []

print(f"Executing Athena query on 'whatsappchatmessages' (isUser=True) for {len(lead_list_wa):,} leads across {len(lead_batches)} batch(es)...")

for idx, batch in enumerate(lead_batches, 1):
    batch_ids_str = ", ".join(f"'{l}'" for l in batch)
    
    # Query whatsappchatmessages table for isUser == True
    sql_query = f"""
        SELECT 
            lead AS lead_oid,
            time
        FROM "leadsdb-prod"."whatsappchatmessages" 
        WHERE (isuser = 'true' OR isuser = 'True' OR isuser = '1')
          AND lead IN ({batch_ids_str})
    """
    
    batch_df = wr.athena.read_sql_query(
        sql=sql_query,
        database="leadsdb-prod",
        boto3_session=session,
        workgroup="metabase",
        s3_output="s3://metabase-asbl/athena-query-results/",
        ctas_approach=False,
    )
    wa_parts.append(batch_df)
    if idx % 50 == 0 or idx == len(lead_batches):
        print(f"  • Batch {idx}/{len(lead_batches)} complete ({sum(len(d) for d in wa_parts):,} total rows)...")

# 3. Combine batches into wa DataFrame
wa = pd.concat(wa_parts, ignore_index=True) if wa_parts else pd.DataFrame(columns=["lead_oid", "time"])

# 4. Calculate WhatsApp Inbound Stats
if wa.empty:
    wa_stats = pd.DataFrame(columns=["lead_oid", "total_whatsapp_inbound", "last_whatsapp_inbound", "days_since_whatsapp_inbound"])
else:
    wa["lead_oid"] = wa["lead_oid"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    wa["time_ist"] = pd.to_datetime(wa["time"], errors="coerce")
    
    # Standardize to timezone-naive IST
    if hasattr(wa["time_ist"].dt, "tz_localize"):
        try:
            wa["time_ist"] = wa["time_ist"].dt.tz_localize(None)
        except Exception:
            wa["time_ist"] = wa["time_ist"].dt.tz_convert(None)
            
    wa_stats = (
        wa.groupby("lead_oid", as_index=False)
        .agg(
            total_whatsapp_inbound=("time_ist", "size"),
            last_whatsapp_inbound=("time_ist", "max"),
        )
    )
    now_ist = pd.Timestamp.now().floor("s")
    wa_stats["days_since_whatsapp_inbound"] = (now_ist - wa_stats["last_whatsapp_inbound"]).dt.total_seconds() / 86400.0

# 5. Merge wa_stats into df1
df5_clean = df5.copy()
df5_clean["lead_oid"] = df5_clean["lead"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

df5_with_wa = df5_clean.merge(
    wa_stats[["lead_oid", "total_whatsapp_inbound", "last_whatsapp_inbound", "days_since_whatsapp_inbound"]],
    on="lead_oid",
    how="left"
)

# 6. Fill missing values (0 for total inbound count)
df5_with_wa["total_whatsapp_inbound"] = (
    df5_with_wa["total_whatsapp_inbound"].fillna(0).astype(int)
)

print("=" * 75)
print(f"🎉 SUCCESS! WHATSAPP INBOUND STATS MERGED SUCCESSFULLY!")
print(f"   Leads with Inbound > 0: {(df5_with_wa['total_whatsapp_inbound'] > 0).sum():,} / {len(df5_with_wa):,} rows")
print("=" * 75)
df5_with_wa[["lead", "total_whatsapp_inbound", "days_since_whatsapp_inbound"]].head(20)


Executing Athena query on 'whatsappchatmessages' (isUser=True) for 105,796 leads across 212 batch(es)...
  • Batch 50/212 complete (1,983 total rows)...
  • Batch 100/212 complete (2,266 total rows)...
  • Batch 150/212 complete (2,540 total rows)...
  • Batch 200/212 complete (3,624 total rows)...
  • Batch 212/212 complete (3,944 total rows)...
🎉 SUCCESS! WHATSAPP INBOUND STATS MERGED SUCCESSFULLY!
   Leads with Inbound > 0: 2,543 / 111,738 rows


,lead,total_whatsapp_inbound,days_since_whatsapp_inbound
0,6a023a53d90c4857b766189c,0,NaN
1,6a023b888b2d7b451592811f,0,NaN
2,6a023bb0f858b6e164691c23,0,NaN
3,6a023c83ddaa0af3065f4e91,0,NaN
4,6a023d25d90c4857b7661b5d,0,NaN
5,6a023d3d8b2d7b45159282d0,0,NaN
6,6a023e02f858b6e164691e26,0,NaN
7,6a023f6398ffe523164caa45,0,NaN
8,6a023ffef7f146ffde9fe840,0,NaN
9,6a023ffef7f146ffde9fe840,0,NaN


In [687]:
# Extract df_wa_raw directly from the `wa` DataFrame already in memory
df_wa_raw = wa[['lead_oid', 'time', 'time_ist']].copy().rename(columns={'lead_oid': 'lead', 'time_ist': 'time_dt'})

print("=" * 75)
print(f"🎉 SUCCESS! Extracted {len(df_wa_raw):,} RAW individual WhatsApp records directly from memory into `df_wa_raw`")
print("=" * 75)
print("DataFrame Columns:", df_wa_raw.columns.tolist())
df_wa_raw.head(10)


🎉 SUCCESS! Extracted 3,944 RAW individual WhatsApp records directly from memory into `df_wa_raw`
DataFrame Columns: ['lead', 'time', 'time_dt']


,lead,time,time_dt
0,6a029b19d90c4857b76665e2,2026-05-27 15:14:58.776000,2026-05-27 15:14:58.776
1,6a024452f7f146ffde9feae7,2026-05-17 09:26:21.010000,2026-05-17 09:26:21.010
2,6a02a861ddaa0af3065fb9c4,2026-05-14 13:52:27.313000,2026-05-14 13:52:27.313
3,6a02a14bf7f146ffdea060e6,2026-05-19 01:22:16.316000,2026-05-19 01:22:16.316
4,6a02e20d2b1368679b7ce46e,2026-05-15 13:48:35.447000,2026-05-15 13:48:35.447
5,6a02e20d2b1368679b7ce46e,2026-05-15 13:48:43.084000,2026-05-15 13:48:43.084
6,6a02e20d2b1368679b7ce46e,2026-05-15 13:48:53.713000,2026-05-15 13:48:53.713
7,6a02e20d2b1368679b7ce46e,2026-05-15 13:48:59.130000,2026-05-15 13:48:59.130
8,6a02e20d2b1368679b7ce46e,2026-05-15 13:49:06.941000,2026-05-15 13:49:06.941
9,6a0290d08b2d7b451592bc20,2026-05-15 17:09:48.153000,2026-05-15 17:09:48.153


,_id,visitorId,globalId,cummulativeTimeSpent,createdAt,instanceNumber,pages,priceSheetsOpened,whatsappClicked,botLog,trackEvents
0,6a631156babc5fbc77dd1cb9,Visitor-7013554,<NA>,0.0,2026-07-24 07:16:38.800,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a631156babc5fbc77dd1cb9,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
1,6a631156f3fe2faf465e10c3,Visitor-7013554,<NA>,0.0,2026-07-24 07:16:38.801,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a631156f3fe2faf465e10c3,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
2,6a6311ad630bca1f2039f763,Visitor-7013554,<NA>,0.0,2026-07-24 07:18:05.080,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a6311ad630bca1f2039f763,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
3,6a6311c11d625ab3f59c5a60,Visitor-7013554,<NA>,0.0,2026-07-24 07:18:25.190,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a6311c11d625ab3f59c5a60,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
4,6a670fb1399e0baa8e4e0aab,Visitor-7013554,<NA>,0.0,2026-07-27 07:58:41.587,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a670fb1399e0baa8e4e0aab,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
...,...,...,...,...,...,...,...,...,...,...,...
43959,6a7a98da692ebf17ecc202af,Visitor-27966709,<NA>,0.0,2026-08-11 03:36:58.637,5,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a7a98da692ebf17ecc202af,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
43960,6a7a9d01692ebf17ecc23c28,Visitor-27969637,<NA>,0.0,2026-08-11 03:54:41.302,4,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a7a9d01692ebf17ecc23c28,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
43961,6a7ad70a0a04bc5f58c97e79,Visitor-26294414,<NA>,0.0,2026-08-11 08:02:18.865,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a7ad70a0a04bc5f58c97e79,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."
43962,6a7ae679631d3293fbd62816,Visitor-27974923,<NA>,0.0,2026-08-11 09:08:09.522,5,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a7ae679631d3293fbd62816,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t..."


In [696]:
import json
from collections import Counter
import pandas as pd

# 1. Collect frequency of every unique eventName across all rows in df7
event_counter = Counter()

# Dynamically resolve event column in df7
event_col = 'trackEvents' if 'trackEvents' in df7.columns else ('trackevents' if 'trackevents' in df7.columns else df7.columns[0])

for val in df7[event_col]:
    # Parse JSON string if needed
    if isinstance(val, str) and val.strip():
        try:
            val = json.loads(val)
        except Exception:
            val = []
            
    if isinstance(val, list):
        for item in val:
            if isinstance(item, dict) and 'eventName' in item:
                ev_list = item['eventName']
                if isinstance(ev_list, str):
                    ev_list = [ev_list]
                elif not isinstance(ev_list, list):
                    continue
                for ev in ev_list:
                    if isinstance(ev, str) and ev.strip():
                        event_counter[ev.strip()] += 1

# 2. Convert to DataFrame
if len(event_counter) > 0:
    df_all_events = pd.DataFrame([
        {'eventName': ev, 'total_hits': cnt}
        for ev, cnt in event_counter.items()
    ]).sort_values(by='total_hits', ascending=False).reset_index(drop=True)
else:
    df_all_events = pd.DataFrame(columns=['eventName', 'total_hits'])

print(f"✅ TOTAL UNIQUE `eventName` STRINGS IN DATASET: {len(df_all_events):,}\n")

# 3. Categorize & Audit Top Events for 100% Confidence
if len(df_all_events) > 0:
    categories = {
        'SITE VISIT BOOKING': ['SITE_VISIT', 'BOOK', 'FLAT', 'VISIT'],
        'WHATSAPP BUTTON': ['WHATSAPP', 'WA_'],
        'BROCHURE DOWNLOAD': ['BROCHURE', 'DOWNLOAD', 'PDF'],
        'PRICE SHEET VIEW': ['PRICE', 'PRICESHEET'],
        'COST SHEET CLICK': ['COST_SHEET', 'COST']
    }

    for group_name, kws in categories.items():
        matched = df_all_events[df_all_events['eventName'].apply(lambda x: any(kw in str(x).upper() for kw in kws))]
        print("=" * 80)
        print(f"📊 EVENT CATEGORY AUDIT: [{group_name}]")
        print(f"   Unique Event Strings: {len(matched):,} | Total Hits: {matched['total_hits'].sum():,}")
        print("=" * 80)
        if len(matched) > 0:
            print(matched.head(15).to_string(index=False))
        else:
            print("No matching event names found.")
        print("\n")
else:
    print("⚠️ No trackEvents entries found in df7.")


✅ TOTAL UNIQUE `eventName` STRINGS IN DATASET: 3,590

📊 EVENT CATEGORY AUDIT: [SITE VISIT BOOKING]
   Unique Event Strings: 335 | Total Hits: 22,885
                                                          eventName  total_hits
                                             HOME$BOOK_A_SITE_VISIT        5197
                                   BOTTOM_BAR$BOOK_A_SITE_VISIT_BTN        3567
           HOME$BOOK_A_SITE_VISIT$BOOK_A_SITE_VISIT$SUBMIT_LEAD_BTN        1811
               HOME$BOOK_A_SITE_VISIT$BOOK_A_SITE_VISIT$GET_OTP_BTN        1238
        PLANS$UNIT_PLAN_SECTION$SITE_PLAN$FLAT_DETAILS$DIALOG_CLOSE        1132
              HOME$BOOK_A_SITE_VISIT$BOOK_A_SITE_VISIT$LEAD_CREATED        1066
                                         HOME$BOOK_A_SITE_VISIT_BTN         838
                                            PRICE$BOOK_A_SITE_VISIT         631
                               SPECIFICATIONS$FLAT$FLOORING_SECTION         356
         PLANS$UNIT_PLAN_SECTION$SITE_PLAN$FLAT_DET

In [697]:
def parse_pages_json(val):
    default_pages = {"home": 0.0, "specifications": 0.0, "plan": 0.0, "location": 0.0, "price": 0.0, "media": 0.0, "amenities": 0.0}
    if isinstance(val, str) and val.strip():
        try:
            parsed = json.loads(val)
            if isinstance(parsed, dict):
                for k, v in parsed.items():
                    if k.lower() in default_pages:
                        default_pages[k.lower()] = float(v) if pd.notnull(v) else 0.0
        except Exception:
            pass
    return default_pages

# Example: Expand pages JSON in df6
pages_df = pd.DataFrame(df6['pages'].apply(parse_pages_json).tolist())


In [713]:
import json
import time
import pandas as pd
import numpy as np

print("🚀 Starting Row-Wise Population for `df6`...")
t0 = time.time()

# ==============================================================================
# STEP 1: PARSE JSON `pages` COLUMN IN df6
# ==============================================================================
print("\n[1/2] Parsing JSON `pages` Column into Row-Wise Page Timespents & Flags...")

def parse_pages_json(val):
    d = {
        "home_timespent": 0.0, 
        "price_timespent": 0.0, 
        "plan_timespent": 0.0, 
        "location_timespent": 0.0, 
        "amenities_timespent": 0.0, 
        "media_timespent": 0.0, 
        "specifications_timespent": 0.0
    }
    if isinstance(val, str) and val.strip():
        try:
            p = json.loads(val)
            if isinstance(p, dict):
                for k, v in p.items():
                    key = f"{k.lower().strip()}_timespent"
                    if key in d:
                        d[key] = float(v) if pd.notnull(v) else 0.0
        except Exception:
            pass
    return d

# Expand JSON pages into individual columns
pages_parsed_df = pd.DataFrame(df6['pages'].apply(parse_pages_json).tolist())

# Generate boolean active flags ({0, 1}) for each page
for col in pages_parsed_df.columns:
    page_name = col.replace("_timespent", "")
    pages_parsed_df[f"is_{page_name}"] = (pages_parsed_df[col] > 0).astype(int)

# Attach page columns to df6
df6_populated = pd.concat([df6.reset_index(drop=True), pages_parsed_df.reset_index(drop=True)], axis=1)

# ==============================================================================
# STEP 2: PARSE ROW-WISE ACTION CLICK FLAGS FROM trackEvents (df7)
# ==============================================================================
print("\n[2/2] Extracting Row-Wise Action Click Flags from `df7['trackEvents']`...")

def parse_track_events_json(val):
    flags = {
        'is_site_visit_book_clicked': 0,
        'is_whatsapp_clicked': 0,
        'is_brochure_downloaded': 0,
        'is_price_sheet_viewed': 0,
        'is_cost_sheet_clicked': 0
    }
    
    if isinstance(val, str) and val.strip():
        try:
            val = json.loads(val)
        except Exception:
            val = []
            
    if not isinstance(val, list):
        return flags
        
    for item in val:
        if isinstance(item, dict) and 'eventName' in item:
            ev_list = item['eventName']
            if isinstance(ev_list, str):
                ev_list = [ev_list]
            elif not isinstance(ev_list, list):
                continue
                
            for ev in ev_list:
                if not isinstance(ev, str):
                    continue
                u = ev.upper()
                
                # 1. Site Visit Booking Clicks
                if any(k in u for k in ['BOOK_A_SITE_VISIT', 'SITE_VISIT_BTN', 'START_SITE_VISIT']):
                    flags['is_site_visit_book_clicked'] = 1
                # 2. WhatsApp Button Clicks
                if any(k in u for k in ['WHATSAPP', 'WA_']):
                    flags['is_whatsapp_clicked'] = 1
                # 3. Brochure Downloads
                if any(k in u for k in ['BROCHURE', 'DOWNLOAD', 'PDF']):
                    flags['is_brochure_downloaded'] = 1
                # 4. Price Sheet Views
                if any(k in u for k in ['PRICE_SHEET', 'PRICE_SECTION', 'PRICESHEET', 'PRICE_PARTICULARS', 'UNLOCK_PRICE']):
                    flags['is_price_sheet_viewed'] = 1
                # 5. Cost Sheet Clicks
                if any(k in u for k in ['COST_SHEET', 'SHOW_COST_SHEET', 'VIEW_COST_SHEET']):
                    flags['is_cost_sheet_clicked'] = 1

    return flags

# Extract action click flags from df7
if 'df7' in globals() and 'trackEvents' in df7.columns:
    event_flags_df = pd.DataFrame(df7['trackEvents'].apply(parse_track_events_json).tolist())
    df7_events = pd.concat([df7[['botLog']].reset_index(drop=True), event_flags_df.reset_index(drop=True)], axis=1)
    
    # Take max per botLog ID
    df7_events_grouped = df7_events.groupby('botLog').max().reset_index()
    
    # Merge action click flags onto df6_populated (matching df6['_id'] == df7['botLog'])
    df6_populated['botLog_clean'] = df6_populated['_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    df7_events_grouped['botLog_clean'] = df7_events_grouped['botLog'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    
    df6_populated = df6_populated.merge(
        df7_events_grouped.drop(columns=['botLog']),
        on='botLog_clean',
        how='left'
    ).drop(columns=['botLog_clean'], errors='ignore')

# Fill missing click flags with 0
click_flag_cols = ['is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked']
for col in click_flag_cols:
    if col in df6_populated.columns:
        df6_populated[col] = df6_populated[col].fillna(0).astype(int)

t1 = time.time()

print("=" * 80)
print(f"🎉 `df6_populated` CREATED SUCCESSFULLY IN {t1 - t0:.2f} SECONDS!")
print("=" * 80)
print(f"  • Total Rows in df6_populated:    {len(df6_populated):,}")
print(f"  • Total Columns in df6_populated: {len(df6_populated.columns)}")
print("=" * 80)
print("DataFrame Columns:", df6_populated.columns.tolist())
df6_populated.head(10)


🚀 Starting Row-Wise Population for `df6`...

[1/2] Parsing JSON `pages` Column into Row-Wise Page Timespents & Flags...

[2/2] Extracting Row-Wise Action Click Flags from `df7['trackEvents']`...
🎉 `df6_populated` CREATED SUCCESSFULLY IN 3.51 SECONDS!
  • Total Rows in df6_populated:    43,964
  • Total Columns in df6_populated: 30
DataFrame Columns: ['_id', 'visitorId', 'globalId', 'cummulativeTimeSpent', 'createdAt', 'instanceNumber', 'pages', 'priceSheetsOpened', 'whatsappClicked', 'botLog', 'trackEvents', 'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent', 'amenities_timespent', 'media_timespent', 'specifications_timespent', 'is_home', 'is_price', 'is_plan', 'is_location', 'is_amenities', 'is_media', 'is_specifications', 'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked']


,_id,visitorId,globalId,cummulativeTimeSpent,createdAt,instanceNumber,pages,priceSheetsOpened,whatsappClicked,botLog,trackEvents,home_timespent,price_timespent,plan_timespent,location_timespent,amenities_timespent,media_timespent,specifications_timespent,is_home,is_price,is_plan,is_location,is_amenities,is_media,is_specifications,is_site_visit_book_clicked,is_whatsapp_clicked,is_brochure_downloaded,is_price_sheet_viewed,is_cost_sheet_clicked
0,6a631156babc5fbc77dd1cb9,Visitor-7013554,<NA>,0.0,2026-07-24 07:16:38.800,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a631156babc5fbc77dd1cb9,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
1,6a631156f3fe2faf465e10c3,Visitor-7013554,<NA>,0.0,2026-07-24 07:16:38.801,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a631156f3fe2faf465e10c3,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
2,6a6311ad630bca1f2039f763,Visitor-7013554,<NA>,0.0,2026-07-24 07:18:05.080,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a6311ad630bca1f2039f763,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
3,6a6311c11d625ab3f59c5a60,Visitor-7013554,<NA>,0.0,2026-07-24 07:18:25.190,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a6311c11d625ab3f59c5a60,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
4,6a670fb1399e0baa8e4e0aab,Visitor-7013554,<NA>,0.0,2026-07-27 07:58:41.587,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a670fb1399e0baa8e4e0aab,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
5,6a670fb1188b95f6cba3a242,Visitor-7013554,<NA>,0.0,2026-07-27 07:58:41.589,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a670fb1188b95f6cba3a242,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
6,6a670fb16e1b0c8b4f9ae222,Visitor-7013554,<NA>,0.0,2026-07-27 07:58:41.770,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a670fb16e1b0c8b4f9ae222,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
7,6a686b8c99511008ddbe5f9a,Visitor-7013554,<NA>,0.0,2026-07-28 08:42:52.668,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a686b8c99511008ddbe5f9a,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,1,0,0
8,6a6af3ed6036da8ee3891a8f,Visitor-7013554,<NA>,0.0,2026-07-30 06:49:17.018,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a6af3ed6036da8ee3891a8f,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
9,6a6af3ed93a7989d675f3a29,Visitor-7013554,<NA>,0.0,2026-07-30 06:49:17.105,1,"{""home"": 0, ""specifications"": 0, ""plan"": 0, ""l...",<NA>,<NA>,6a6af3ed93a7989d675f3a29,"[{""eventName"": [""URL_HIT""], ""pageName"": """", ""t...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0


In [724]:
_leadsStageDF_3m=_leadsStageDF_6m[_leadsStageDF_6m['lead'].isin(df5['lead'].tolist())]

In [816]:
df55=df5.copy()

In [ ]:
import pandas as pd
import numpy as np

print("🚀 Updating `episode_start_time` for continuous episode coverage...")

# 1. Clean datetime types
df5['episode_start_time'] = pd.to_datetime(df5['episode_start_time'], errors='coerce')
df5['episode_end_time'] = pd.to_datetime(df5['episode_end_time'], errors='coerce')

# Ensure chronological sorting by lead and episode_number
df5 = df5.sort_values(by=['lead', 'episode_number']).reset_index(drop=True)

# 2. Get episode_end_time of episode (x-1) for each lead
df5['prev_ep_end_time'] = df5.groupby('lead')['episode_end_time'].shift(1)

# 3. Apply offset: start_time(x) = end_time(x-1) + 1 millisecond (for x > 1)
micro_offset = pd.Timedelta(milliseconds=1)
df5['episode_start_time'] = np.where(
    df5['episode_number'] > 1,
    df5['prev_ep_end_time'] + micro_offset,
    df5['episode_start_time']
)

df5['episode_start_time'] = pd.to_datetime(df5['episode_start_time'])
df5.drop(columns=['prev_ep_end_time'], errors='ignore', inplace=True)

# Also update in df_master_episode_dataset if present
if 'df_master_episode_dataset' in globals():
    df_master_episode_dataset['episode_start_time'] = pd.to_datetime(df_master_episode_dataset['episode_start_time'], errors='coerce')
    df_master_episode_dataset['episode_end_time'] = pd.to_datetime(df_master_episode_dataset['episode_end_time'], errors='coerce')
    df_master_episode_dataset = df_master_episode_dataset.sort_values(by=['lead', 'episode_number']).reset_index(drop=True)
    
    df_master_episode_dataset['prev_ep_end_time'] = df_master_episode_dataset.groupby('lead')['episode_end_time'].shift(1)
    df_master_episode_dataset['episode_start_time'] = np.where(
        df_master_episode_dataset['episode_number'] > 1,
        df_master_episode_dataset['prev_ep_end_time'] + micro_offset,
        df_master_episode_dataset['episode_start_time']
    )
    df_master_episode_dataset['episode_start_time'] = pd.to_datetime(df_master_episode_dataset['episode_start_time'])
    df_master_episode_dataset.drop(columns=['prev_ep_end_time'], errors='ignore', inplace=True)

print("=" * 80)
print("🎉 SUCCESS! `episode_start_time` ADJUSTED CONTINUOUSLY ACROSS ALL EPISODES!")
print("=" * 80)
df5[['lead', 'episode_number', 'episode_id', 'episode_start_time', 'episode_end_time']].head(10)


In [825]:
import time
import pandas as pd
import numpy as np

print("🚀 Starting Updated Episode Web Feature Aggregation...")
t0 = time.time()

# 1. Ensure datetime formatting and clean visitorId keys
df5_base = df5.copy()
df5_base['episode_start_time'] = pd.to_datetime(df5_base['episode_start_time'], errors='coerce')
df5_base['episode_end_time'] = pd.to_datetime(df5_base['episode_end_time'], errors='coerce')
df5_base['visitorId_clean'] = df5_base['visitorId'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df6_pop = df6_populated.copy()
df6_pop['createdAt_dt'] = pd.to_datetime(df6_pop['createdAt'], errors='coerce')
df6_pop['visitorId_clean'] = df6_pop['visitorId'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# 2. Merge df5_base with df6_pop on visitorId_clean
m_web = df5_base[['episode_id', 'visitorId_clean', 'episode_start_time', 'episode_end_time']].merge(
    df6_pop, 
    on='visitorId_clean', 
    how='inner'
)

# 3. Filter strictly within time window: episode_start_time <= createdAt_dt <= episode_end_time
v_web = m_web[
    (m_web['createdAt_dt'] >= m_web['episode_start_time']) & 
    (m_web['createdAt_dt'] <= m_web['episode_end_time'])
].copy()

print(f"  • Matched Web Visits in Episode Windows: {len(v_web):,} rows")

# 4. Perform Episode Aggregations
if not v_web.empty:
    v_web['cummulativeTimeSpent'] = pd.to_numeric(v_web.get('cummulativeTimeSpent', 0), errors='coerce').fillna(0.0)
    v_web['instanceNumber'] = pd.to_numeric(v_web.get('instanceNumber', 0), errors='coerce').fillna(0).astype(int)
    
    page_ts_cols = ['home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent', 'amenities_timespent', 'media_timespent', 'specifications_timespent']
    for c in page_ts_cols:
        if c in v_web.columns: v_web[c] = pd.to_numeric(v_web[c], errors='coerce').fillna(0.0)
            
    action_flag_cols = ['is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked']
    for c in action_flag_cols:
        if c in v_web.columns: v_web[c] = pd.to_numeric(v_web[c], errors='coerce').fillna(0).astype(int)

    agg_dict = {
        'total_web_timespent': ('cummulativeTimeSpent', 'sum'),
        'instance_min': ('instanceNumber', 'min'),
        'instance_max': ('instanceNumber', 'max'),
        'active_session_count': ('cummulativeTimeSpent', lambda s: int((s > 0).sum())),
    }
    
    for c in page_ts_cols:
        if c in v_web.columns: agg_dict[c] = (c, 'sum')
            
    for c in action_flag_cols:
        if c in v_web.columns: agg_dict[c] = (c, lambda s: int((s > 0).any()))

    web_ep_agg = v_web.groupby('episode_id').agg(**agg_dict).reset_index()
    
    # Calculate total_session_count = max(instanceNumber) - min(instanceNumber) + 1
    web_ep_agg['total_session_count'] = web_ep_agg['instance_max'] - web_ep_agg['instance_min'] + 1
    web_ep_agg.drop(columns=['instance_min', 'instance_max'], inplace=True)

    df5_with_web = df5_base.drop(columns=['visitorId_clean'], errors='ignore').merge(
        web_ep_agg, 
        on='episode_id', 
        how='left'
    )
else:
    df5_with_web = df5_base.drop(columns=['visitorId_clean'], errors='ignore').copy()

# 5. Fill defaults for episodes with 0 web visits
default_numerics = ['total_web_timespent', 'total_session_count', 'active_session_count'] + [c for c in page_ts_cols if c in df6_populated.columns]
for col in default_numerics:
    if col not in df5_with_web.columns:
        df5_with_web[col] = 0.0
    else:
        df5_with_web[col] = df5_with_web[col].fillna(0.0)

default_flags = [c for c in action_flag_cols if c in df6_populated.columns]
for col in default_flags:
    if col not in df5_with_web.columns:
        df5_with_web[col] = 0
    else:
        df5_with_web[col] = df5_with_web[col].fillna(0).astype(int)

t1 = time.time()

print("=" * 80)
print(f"🎉 SUCCESS! EPISODE WEB AGGREGATION COMPLETE ({t1 - t0:.2f} SECONDS)!")
print("=" * 80)
print(f"  • Total Episode Rows:                                {len(df5_with_web):,}")
print(f"  • Episodes with Web Activity (>0 total timespent):   {(df5_with_web['total_web_timespent'] > 0).sum():,}")
print(f"  • Episodes with Active Sessions (>0 active count):   {(df5_with_web['active_session_count'] > 0).sum():,}")
print("=" * 80)
print("DataFrame Columns:", df5_with_web.columns.tolist())
df5_with_web.head(10)


🚀 Starting Updated Episode Web Feature Aggregation...
  • Matched Web Visits in Episode Windows: 11,229 rows
🎉 SUCCESS! EPISODE WEB AGGREGATION COMPLETE (1.40 SECONDS)!
  • Total Episode Rows:                                111,738
  • Episodes with Web Activity (>0 total timespent):   3,767
  • Episodes with Active Sessions (>0 active count):   3,767
DataFrame Columns: ['lead', 'episode_number', 'episode_start_time', 'episode_end_time', 'start_stage', 'end_stage', 'total_stage_movements', 'episode_id', 'is_reactivated_episode', 'masterLeadId', 'bornSource', 'source', 'secondarySource', 'project', 'status', 'lead_createdAt', 'globalId', 'leadId', 'visitorId', 'total_web_timespent', 'active_session_count', 'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent', 'amenities_timespent', 'media_timespent', 'specifications_timespent', 'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked', 'total_se

,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId,visitorId,total_web_timespent,active_session_count,home_timespent,price_timespent,plan_timespent,location_timespent,amenities_timespent,media_timespent,specifications_timespent,is_site_visit_book_clicked,is_whatsapp_clicked,is_brochure_downloaded,is_price_sheet_viewed,is_cost_sheet_clicked,total_session_count
0,6a023a53d90c4857b766189c,1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,New Lead,Not Interested,15,6a023a53d90c4857b766189c_1,0,920554,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:21:39.461,22787547,1177568,Visitor-26920396,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
1,6a023b888b2d7b451592811f,1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,New Lead,Not Interested,2,6a023b888b2d7b451592811f_1,0,920555,66d6b121137a13b8864d16a2,66d6b121137a13b8864d16a2,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:26:48.757,<NA>,1177569,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
2,6a023bb0f858b6e164691c23,1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,New Lead,Not Interested,9,6a023bb0f858b6e164691c23_1,0,920556,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:27:28.565,<NA>,1177570,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
3,6a023c83ddaa0af3065f4e91,1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,New Lead,Not Interested,7,6a023c83ddaa0af3065f4e91_1,0,920557,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:30:59.331,<NA>,1177571,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
4,6a023d25d90c4857b7661b5d,1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,New Lead,Not Interested,10,6a023d25d90c4857b7661b5d_1,0,801227,69d4979d60cfeeb29bcd905b,69d4979d60cfeeb29bcd905b,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:33:41.149,<NA>,1177572,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
5,6a023d3d8b2d7b45159282d0,1,2026-05-11 20:34:06.054,2026-05-12 04:58:20.588,New Lead,Not Interested,2,6a023d3d8b2d7b45159282d0_1,0,920558,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:34:05.772,22787629,1177573,Visitor-26920486,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
6,6a023e02f858b6e164691e26,1,2026-05-11 20:37:24.791,2026-05-12 06:34:14.861,New Lead,Not Interested,5,6a023e02f858b6e164691e26_1,0,920557,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-05-11 20:37:22.788,22787618,1177574,Visitor-26920472,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,2.0
7,6a023f6398ffe523164caa45,1,2026-05-11 20:43:16.163,2026-05-12 07:47:43.976,New Lead,Not Interested,4,6a023f6398ffe523164caa45_1,0,920559,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:43:15.726,<NA>,1177575,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0
8,6a023ffef7f146ffde9fe840,1,2026-05-11 20:45:50.862,2026-05-17 13:14:21.269,New Lead,Not Interested,9,6a023ffef7f146ffde9fe840_1,1,920560,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,"[{""source"": ""64a7859a0fcd685ef70d6dd8"", ""date""...",64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:45:50.600,22787688,1177576,Visitor-26920546,1.155,1.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,3.0
9,6a023ffef7f146ffde9fe840,2,2026-05-17 13:14:21.270,2026-05-23 05:06:00.495,Lead Initiated,Not Interested,15,6a023ffef7f146ffde9fe840_2,0,920560,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,"[{""source"": ""64a7859a0fcd685ef70d6dd8"", ""date""...",64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:45:50.600,22787688,1177576,Visitor-26920546,2009.661,4.0,302.023,0.0,1644.018,0.0,50.741,12.879,0.0,0,1,1,1,0,4.0


In [826]:
import json
import time
import pandas as pd

print("🚀 Calculating Resubmission Count per Episode from secondarySource...")
t0 = time.time()

def calculate_episode_resubmissions(row):
    resub_count = 0
    val = row.get('secondarySource')
    
    start_t = pd.to_datetime(row.get('episode_start_time'), errors='coerce')
    end_t = pd.to_datetime(row.get('episode_end_time'), errors='coerce')
    
    if pd.isnull(start_t) or pd.isnull(end_t):
        return 0
        
    if isinstance(val, str) and val.strip() and val != '[]':
        try:
            parsed = json.loads(val)
            if isinstance(parsed, list):
                for item in parsed:
                    if isinstance(item, dict):
                        is_react = item.get('isReactivation')
                        
                        # Filter strictly for isReactivation == False
                        if is_react is False or str(is_react).lower() == 'false':
                            rec_date = pd.to_datetime(item.get('date'), errors='coerce')
                            
                            # Check date window: episode_start_time <= date <= episode_end_time
                            if pd.notnull(rec_date) and (start_t <= rec_date <= end_t):
                                resub_count += 1
        except Exception:
            pass
            
    return resub_count

# Calculate resubmission count per episode
df5_with_web['resubmission_count_in_episode'] = df5_with_web.apply(calculate_episode_resubmissions, axis=1)
df5_with_web['is_resubmitted_in_episode'] = (df5_with_web['resubmission_count_in_episode'] > 0).astype(int)

t1 = time.time()

print("=" * 80)
print(f"🎉 SUCCESS! RESUBMISSION METRICS CALCULATED IN {t1 - t0:.2f} SECONDS!")
print("=" * 80)
print(f"  • Total Episodes:                                       {len(df5_with_web):,}")
print(f"  • Episodes with >0 Resubmissions (is_resubmitted = 1):  {df5_with_web['is_resubmitted_in_episode'].sum():,} ({(df5_with_web['is_resubmitted_in_episode'].sum()/len(df5_with_web))*100:.2f}%)")
print("=" * 80)

df5_with_web[['episode_id', 'episode_start_time', 'episode_end_time', 'resubmission_count_in_episode', 'is_resubmitted_in_episode']].head(15)


🚀 Calculating Resubmission Count per Episode from secondarySource...
🎉 SUCCESS! RESUBMISSION METRICS CALCULATED IN 2.59 SECONDS!
  • Total Episodes:                                       111,738
  • Episodes with >0 Resubmissions (is_resubmitted = 1):  6,577 (5.89%)


,episode_id,episode_start_time,episode_end_time,resubmission_count_in_episode,is_resubmitted_in_episode
0,6a023a53d90c4857b766189c_1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,0,0
1,6a023b888b2d7b451592811f_1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,0,0
2,6a023bb0f858b6e164691c23_1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,0,0
3,6a023c83ddaa0af3065f4e91_1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,0,0
4,6a023d25d90c4857b7661b5d_1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,0,0
5,6a023d3d8b2d7b45159282d0_1,2026-05-11 20:34:06.054,2026-05-12 04:58:20.588,0,0
6,6a023e02f858b6e164691e26_1,2026-05-11 20:37:24.791,2026-05-12 06:34:14.861,0,0
7,6a023f6398ffe523164caa45_1,2026-05-11 20:43:16.163,2026-05-12 07:47:43.976,0,0
8,6a023ffef7f146ffde9fe840_1,2026-05-11 20:45:50.862,2026-05-17 13:14:21.269,0,0
9,6a023ffef7f146ffde9fe840_2,2026-05-17 13:14:21.270,2026-05-23 05:06:00.495,0,0


In [827]:
import time
import pandas as pd
import numpy as np

print("🚀 Calculating Intra-Episode Stage Metrics from Stage Histories...")
t0 = time.time()

# 1. Clean datetime and lead keys for df5_with_web and stage history table
df_base = df5_with_web.copy()
df_base['episode_start_time'] = pd.to_datetime(df_base['episode_start_time'], errors='coerce')
df_base['episode_end_time'] = pd.to_datetime(df_base['episode_end_time'], errors='coerce')
df_base['lead_clean'] = df_base['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Target stage history table (uses _leadsStageDF_6m or _leadsStageDF_3m)
stg_df = _leadsStageDF_6m if '_leadsStageDF_6m' in globals() else (_leadsStageDF_3m if '_leadsStageDF_3m' in globals() else None)

if stg_df is not None:
    stg_clean = stg_df.copy()
    stg_clean['createdAt_dt'] = pd.to_datetime(stg_clean['createdAt'], errors='coerce')
    stg_clean['lead_clean'] = stg_clean['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    stg_clean = stg_clean.dropna(subset=['lead_clean', 'createdAt_dt']).sort_values(by=['lead_clean', 'createdAt_dt'])
    
    # 2. Time-window merge: episode_start_time <= createdAt_dt <= episode_end_time
    m_stg = df_base[['episode_id', 'lead_clean', 'episode_start_time', 'episode_end_time']].merge(
        stg_clean, 
        on='lead_clean', 
        how='inner'
    )
    
    v_stg = m_stg[
        (m_stg['createdAt_dt'] >= m_stg['episode_start_time']) & 
        (m_stg['createdAt_dt'] <= m_stg['episode_end_time'])
    ].copy()
    
    print(f"  • Matched Stage History Records in Episode Windows: {len(v_stg):,} rows")
    
    # 3. Calculate did_reach_site_visit_in_episode (reason is 'Visited Site' or 'Virtual Site Visit')
    v_stg['is_site_visit_reason'] = v_stg['reason'].astype(str).str.strip().str.lower().isin(['visited site', 'virtual site visit'])
    sv_agg = v_stg.groupby('episode_id')['is_site_visit_reason'].any().astype(int).reset_index().rename(
        columns={'is_site_visit_reason': 'did_reach_site_visit_in_episode'}
    )
    
    # 4. Calculate episode_stage_movement_count (isStageMovement is True)
    v_stg['is_mvt'] = v_stg['isStageMovement'].astype(str).str.strip().str.lower().isin(['true', '1'])
    mvt_agg = v_stg.groupby('episode_id')['is_mvt'].sum().astype(int).reset_index().rename(
        columns={'is_mvt': 'episode_stage_movement_count'}
    )
    
    # 5. Extract stage_before_ni (latest stage in episode where stage != 'Not Interested')
    v_non_ni = v_stg[v_stg['stage'].astype(str).str.strip() != 'Not Interested']
    stage_before_ni_agg = v_non_ni.groupby('episode_id')['stage'].last().reset_index().rename(
        columns={'stage': 'stage_before_ni'}
    )
    
    # 6. Extract cancellation_reason_at_episode_end (logged at episode_end_time)
    v_end = v_stg[v_stg['createdAt_dt'] == v_stg['episode_end_time']]
    cancellation_agg = v_end.groupby('episode_id')['cancellation_reason'].first().reset_index().rename(
        columns={'cancellation_reason': 'cancellation_reason_at_episode_end'}
    )
    
    # Merge all stage metrics back onto df_base
    df_base = df_base.merge(sv_agg, on='episode_id', how='left')
    df_base = df_base.merge(mvt_agg, on='episode_id', how='left')
    df_base = df_base.merge(stage_before_ni_agg, on='episode_id', how='left')
    df_base = df_base.merge(cancellation_agg, on='episode_id', how='left')

# 7. Calculate episode_duration_days
df_base['episode_duration_days'] = (df_base['episode_end_time'] - df_base['episode_start_time']).dt.total_seconds() / 86400.0

# Fill defaults
df_base['did_reach_site_visit_in_episode'] = df_base['did_reach_site_visit_in_episode'].fillna(0).astype(int)
df_base['episode_stage_movement_count'] = df_base['episode_stage_movement_count'].fillna(0).astype(int)
df_base['stage_before_ni'] = df_base['stage_before_ni'].fillna(df_base['start_stage'])
df_base['cancellation_reason_at_episode_end'] = df_base['cancellation_reason_at_episode_end'].fillna('None')

if 'lead_clean' in df_base.columns:
    df_base.drop(columns=['lead_clean'], inplace=True)

df_final_episodes = df_base.copy()

t1 = time.time()

print("=" * 80)
print(f"🎉 SUCCESS! EPISODE STAGE METRICS COMPLETED ({t1 - t0:.2f} SECONDS)!")
print("=" * 80)
print(f"  • Total Episode Rows in df_final_episodes: {len(df_final_episodes):,}")
print(f"  • Episodes with Site Visit (did_reach_site_visit = 1): {(df_final_episodes['did_reach_site_visit_in_episode'] == 1).sum():,}")
print("=" * 80)
print("DataFrame Columns:", df_final_episodes.columns.tolist())
df_final_episodes.head(10)


🚀 Calculating Intra-Episode Stage Metrics from Stage Histories...
  • Matched Stage History Records in Episode Windows: 725,950 rows
🎉 SUCCESS! EPISODE STAGE METRICS COMPLETED (4.76 SECONDS)!
  • Total Episode Rows in df_final_episodes: 111,738
  • Episodes with Site Visit (did_reach_site_visit = 1): 1,965
DataFrame Columns: ['lead', 'episode_number', 'episode_start_time', 'episode_end_time', 'start_stage', 'end_stage', 'total_stage_movements', 'episode_id', 'is_reactivated_episode', 'masterLeadId', 'bornSource', 'source', 'secondarySource', 'project', 'status', 'lead_createdAt', 'globalId', 'leadId', 'visitorId', 'total_web_timespent', 'active_session_count', 'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent', 'amenities_timespent', 'media_timespent', 'specifications_timespent', 'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked', 'total_session_count', 'resubmission_count_in_episode',

,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId,visitorId,total_web_timespent,active_session_count,home_timespent,price_timespent,plan_timespent,location_timespent,amenities_timespent,media_timespent,specifications_timespent,is_site_visit_book_clicked,is_whatsapp_clicked,is_brochure_downloaded,is_price_sheet_viewed,is_cost_sheet_clicked,total_session_count,resubmission_count_in_episode,is_resubmitted_in_episode,did_reach_site_visit_in_episode,episode_stage_movement_count,stage_before_ni,cancellation_reason_at_episode_end,episode_duration_days
0,6a023a53d90c4857b766189c,1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,New Lead,Not Interested,15,6a023a53d90c4857b766189c_1,0,920554,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:21:39.461,22787547,1177568,Visitor-26920396,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,64a785845bbdfeeb1a2372cf,5.573329
1,6a023b888b2d7b451592811f,1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,New Lead,Not Interested,2,6a023b888b2d7b451592811f_1,0,920555,66d6b121137a13b8864d16a2,66d6b121137a13b8864d16a2,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:26:48.757,<NA>,1177569,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.033148
2,6a023bb0f858b6e164691c23,1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,New Lead,Not Interested,9,6a023bb0f858b6e164691c23_1,0,920556,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:27:28.565,<NA>,1177570,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad787015a,13.428425
3,6a023c83ddaa0af3065f4e91,1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,New Lead,Not Interested,7,6a023c83ddaa0af3065f4e91_1,0,920557,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:30:59.331,<NA>,1177571,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad7870153,0.549766
4,6a023d25d90c4857b7661b5d,1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,New Lead,Not Interested,10,6a023d25d90c4857b7661b5d_1,0,801227,69d4979d60cfeeb29bcd905b,69d4979d60cfeeb29bcd905b,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:33:41.149,<NA>,1177572,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad787015c,3.601647
5,6a023d3d8b2d7b45159282d0,1,2026-05-11 20:34:06.054,2026-05-12 04:58:20.588,New Lead,Not Interested,2,6a023d3d8b2d7b45159282d0_1,0,920558,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:34:05.772,22787629,1177573,Visitor-26920486,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.350168
6,6a023e02f858b6e164691e26,1,2026-05-11 20:37:24.791,2026-05-12 06:34:14.861,New Lead,Not Interested,5,6a023e02f858b6e164691e26_1,0,920557,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-05-11 20:37:22.788,22787618,1177574,Visitor-26920472,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,2.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.414468
7,6a023f6398ffe523164caa45,1,2026-05-11 20:43:16.163,2026-05-12 07:47:43.976,New Lead,Not Interested,4,6a023f6398ffe523164caa45_1,0,920559,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:43:15.726,<NA>,1177575,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad7870153,0.461433
8,6a023ffef7f146ffde9fe840,1,2026-05-11 20:45:50.862,2026-05-17 13:14:21.269,New Lead,Not Interested,9,6a023ffef7f146ffde9fe840_1,1,920560,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,"

In [828]:
import json
import time
import awswrangler as wr
import boto3
import pandas as pd


# Build Source ID -> Name dictionary
source_dict = dict(zip(
    _leadSourceDF['_id'].astype(str).str.strip(),
    _leadSourceDF['name'].astype(str).str.strip()
))

print(f"  ✅ Loaded {len(source_dict):,} Lead Source Mappings.")

# ------------------------------------------------------------------------------
# 2. Extract Episode Source ID (bornSource for Ep 1, reactivated source for Ep 2+)
# ------------------------------------------------------------------------------

t0 = time.time()

def resolve_episode_source_id(row):
    ep_num = row.get('episode_number')
    born_src = str(row.get('bornSource', '')).strip()
    start_t = pd.to_datetime(row.get('episode_start_time'), errors='coerce')
    val = row.get('secondarySource')
    
    # Episode 1 strictly uses bornSource
    if ep_num == 1 or pd.isnull(start_t):
        return born_src if born_src else 'Unknown'
        
    matching_source = None
    if isinstance(val, str) and val.strip() and val != '[]':
        try:
            parsed = json.loads(val)
            if isinstance(parsed, list):
                # Look for isReactivation == True entry matching episode_start_time
                for item in reversed(parsed):
                    if isinstance(item, dict):
                        is_react = item.get('isReactivation')
                        if is_react is True or str(is_react).lower() == 'true':
                            rec_date = pd.to_datetime(item.get('date'), errors='coerce')
                            # Match date within 1 day of episode_start_time
                            if pd.notnull(rec_date) and abs((rec_date - start_t).total_seconds()) <= 86400:
                                matching_source = str(item.get('source', '')).strip()
                                break
        except Exception:
            pass
            
    return matching_source if matching_source else born_src

# Compute source ID and map to readable name
df_final_episodes['episode_source_id'] = df_final_episodes.apply(resolve_episode_source_id, axis=1)
df_final_episodes['episode_source_name'] = (
    df_final_episodes['episode_source_id']
    .map(source_dict)
    .fillna(df_final_episodes['episode_source_id'])
)

t1 = time.time()

print("=" * 80)
print(f"🎉 SUCCESS! EPISODE SOURCE MAPPED IN {t1 - t0:.2f} SECONDS!")
print("=" * 80)
print(df_final_episodes[['episode_id', 'episode_number', 'episode_source_id', 'episode_source_name']].head(15).to_string(index=False))


  ✅ Loaded 315 Lead Source Mappings.
🎉 SUCCESS! EPISODE SOURCE MAPPED IN 2.63 SECONDS!
                episode_id  episode_number        episode_source_id episode_source_name
6a023a53d90c4857b766189c_1               1 64a7859a0fcd685ef70d6dd8            Web Lead
6a023b888b2d7b451592811f_1               1 66d6b121137a13b8864d16a2           FIM Forms
6a023bb0f858b6e164691c23_1               1 69d8ae93e2ee6bc0cbfe6674      Pre-launch Web
6a023c83ddaa0af3065f4e91_1               1 69d8ae93e2ee6bc0cbfe6674      Pre-launch Web
6a023d25d90c4857b7661b5d_1               1 69d4979d60cfeeb29bcd905b            Whatsapp
6a023d3d8b2d7b45159282d0_1               1 64a7859a0fcd685ef70d6dd8            Web Lead
6a023e02f858b6e164691e26_1               1 66683e8865601d05fa511e02            Web Lead
6a023f6398ffe523164caa45_1               1 69d8ae93e2ee6bc0cbfe6674      Pre-launch Web
6a023ffef7f146ffde9fe840_1               1 64a7859a0fcd685ef70d6dd8            Web Lead
6a023ffef7f146ffde9fe840_2       

In [829]:
import time
import pandas as pd
import numpy as np

print("🚀 Calculating WhatsApp Intra-Episode Features from `df_wa_raw`...")
t0 = time.time()

# 1. Clean datetime and keys
df_base_wa = df_final_episodes.copy()
df_base_wa['episode_start_time'] = pd.to_datetime(df_base_wa['episode_start_time'], errors='coerce')
df_base_wa['episode_end_time'] = pd.to_datetime(df_base_wa['episode_end_time'], errors='coerce')
df_base_wa['lead_clean'] = df_base_wa['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

if 'df_wa_raw' in globals() and not df_wa_raw.empty:
    wa_clean = df_wa_raw.copy()
    
    # Resolve timestamp column
    time_col = 'time_dt' if 'time_dt' in wa_clean.columns else 'time'
    wa_clean['time_dt'] = pd.to_datetime(wa_clean[time_col], errors='coerce')
    
    # Resolve lead column
    lead_col = 'lead' if 'lead' in wa_clean.columns else ('lead_oid' if 'lead_oid' in wa_clean.columns else wa_clean.columns[0])
    wa_clean['lead_clean'] = wa_clean[lead_col].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    
    wa_clean = wa_clean.dropna(subset=['lead_clean', 'time_dt']).sort_values(by=['lead_clean', 'time_dt'])
    
    # 2. Time-window merge: episode_start_time <= time_dt <= episode_end_time
    m_wa = df_base_wa[['episode_id', 'lead_clean', 'episode_start_time', 'episode_end_time']].merge(
        wa_clean, 
        on='lead_clean', 
        how='inner'
    )
    
    v_wa = m_wa[
        (m_wa['time_dt'] >= m_wa['episode_start_time']) & 
        (m_wa['time_dt'] <= m_wa['episode_end_time'])
    ].copy()
    
    print(f"  • Matched WhatsApp Messages in Episode Windows: {len(v_wa):,} rows")
    
    # 3. Perform WhatsApp Aggregations
    if not v_wa.empty:
        wa_ep_agg = v_wa.groupby('episode_id').agg(
            episode_whatsapp_inbound_count=('time_dt', 'count'),
            first_whatsapp_time=('time_dt', 'min'),
            last_whatsapp_time=('time_dt', 'max')
        ).reset_index()
        
        df_base_wa = df_base_wa.merge(wa_ep_agg, on='episode_id', how='left')
    else:
        df_base_wa['episode_whatsapp_inbound_count'] = 0

# 4. Fill defaults and compute recency features
df_base_wa['episode_whatsapp_inbound_count'] = df_base_wa['episode_whatsapp_inbound_count'].fillna(0).astype(int)
df_base_wa['has_whatsapp_inbound_in_episode'] = (df_base_wa['episode_whatsapp_inbound_count'] > 0).astype(int)

# Days from episode_start_time to first WhatsApp inbound message
if 'first_whatsapp_time' in df_base_wa.columns:
    df_base_wa['days_to_first_whatsapp_inbound'] = (
        (df_base_wa['first_whatsapp_time'] - df_base_wa['episode_start_time']).dt.total_seconds() / 86400.0
    ).fillna(-1.0)
    
    df_base_wa['days_to_last_whatsapp_inbound'] = (
        (df_base_wa['last_whatsapp_inbound_time'] if 'last_whatsapp_inbound_time' in df_base_wa.columns else df_base_wa['last_whatsapp_time'] - df_base_wa['episode_start_time']).dt.total_seconds() / 86400.0
    ).fillna(-1.0)
    
    df_base_wa.drop(columns=['first_whatsapp_time', 'last_whatsapp_time'], errors='ignore', inplace=True)
else:
    df_base_wa['days_to_first_whatsapp_inbound'] = -1.0
    df_base_wa['days_to_last_whatsapp_inbound'] = -1.0

if 'lead_clean' in df_base_wa.columns:
    df_base_wa.drop(columns=['lead_clean'], inplace=True)

df_master_episode_dataset = df_base_wa.copy()

t1 = time.time()

print("=" * 80)
print(f"🎉 SUCCESS! MASTER EPISODE DATASET CREATED ({t1 - t0:.2f} SECONDS)!")
print("=" * 80)
print(f"  • Total Episode Rows in Master Dataset:       {len(df_master_episode_dataset):,}")
print(f"  • Episodes with WhatsApp Inbound (>0):        {(df_master_episode_dataset['has_whatsapp_inbound_in_episode'] == 1).sum():,}")
print("=" * 80)
print("Master Dataset Columns:", df_master_episode_dataset.columns.tolist())
df_master_episode_dataset.head(10)


🚀 Calculating WhatsApp Intra-Episode Features from `df_wa_raw`...
  • Matched WhatsApp Messages in Episode Windows: 3,139 rows
🎉 SUCCESS! MASTER EPISODE DATASET CREATED (1.01 SECONDS)!
  • Total Episode Rows in Master Dataset:       111,738
  • Episodes with WhatsApp Inbound (>0):        1,920
Master Dataset Columns: ['lead', 'episode_number', 'episode_start_time', 'episode_end_time', 'start_stage', 'end_stage', 'total_stage_movements', 'episode_id', 'is_reactivated_episode', 'masterLeadId', 'bornSource', 'source', 'secondarySource', 'project', 'status', 'lead_createdAt', 'globalId', 'leadId', 'visitorId', 'total_web_timespent', 'active_session_count', 'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent', 'amenities_timespent', 'media_timespent', 'specifications_timespent', 'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked', 'total_session_count', 'resubmission_count_in_episode', 'is_res

,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId,visitorId,total_web_timespent,active_session_count,home_timespent,price_timespent,plan_timespent,location_timespent,amenities_timespent,media_timespent,specifications_timespent,is_site_visit_book_clicked,is_whatsapp_clicked,is_brochure_downloaded,is_price_sheet_viewed,is_cost_sheet_clicked,total_session_count,resubmission_count_in_episode,is_resubmitted_in_episode,did_reach_site_visit_in_episode,episode_stage_movement_count,stage_before_ni,cancellation_reason_at_episode_end,episode_duration_days,episode_source_id,episode_source_name,episode_whatsapp_inbound_count,has_whatsapp_inbound_in_episode,days_to_first_whatsapp_inbound,days_to_last_whatsapp_inbound
0,6a023a53d90c4857b766189c,1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,New Lead,Not Interested,15,6a023a53d90c4857b766189c_1,0,920554,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:21:39.461,22787547,1177568,Visitor-26920396,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,64a785845bbdfeeb1a2372cf,5.573329,64a7859a0fcd685ef70d6dd8,Web Lead,0,0,-1.0,-1.0
1,6a023b888b2d7b451592811f,1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,New Lead,Not Interested,2,6a023b888b2d7b451592811f_1,0,920555,66d6b121137a13b8864d16a2,66d6b121137a13b8864d16a2,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:26:48.757,<NA>,1177569,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.033148,66d6b121137a13b8864d16a2,FIM Forms,0,0,-1.0,-1.0
2,6a023bb0f858b6e164691c23,1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,New Lead,Not Interested,9,6a023bb0f858b6e164691c23_1,0,920556,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:27:28.565,<NA>,1177570,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad787015a,13.428425,69d8ae93e2ee6bc0cbfe6674,Pre-launch Web,0,0,-1.0,-1.0
3,6a023c83ddaa0af3065f4e91,1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,New Lead,Not Interested,7,6a023c83ddaa0af3065f4e91_1,0,920557,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:30:59.331,<NA>,1177571,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad7870153,0.549766,69d8ae93e2ee6bc0cbfe6674,Pre-launch Web,0,0,-1.0,-1.0
4,6a023d25d90c4857b7661b5d,1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,New Lead,Not Interested,10,6a023d25d90c4857b7661b5d_1,0,801227,69d4979d60cfeeb29bcd905b,69d4979d60cfeeb29bcd905b,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:33:41.149,<NA>,1177572,<NA>,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad787015c,3.601647,69d4979d60cfeeb29bcd905b,Whatsapp,0,0,-1.0,-1.0
5,6a023d3d8b2d7b45159282d0,1,2026-05-11 20:34:06.054,2026-05-12 04:58:20.588,New Lead,Not Interested,2,6a023d3d8b2d7b45159282d0_1,0,920558,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:34:05.772,22787629,1177573,Visitor-26920486,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,0.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.350168,64a7859a0fcd685ef70d6dd8,Web Lead,0,0,-1.0,-1.0
6,6a023e02f858b6e164691e26,1,2026-05-11 20:37:24.791,2026-05-12 06:34:14.861,New Lead,Not Interested,5,6a023e02f858b6e164691e26_1,0,920557,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-05-11 20:37:22.788,22787618,1177574,Visitor-26920472,0.000,0.0,0.000,0.0,0.000,0.0,0.000,0.000,0.0,0,0,0,0,0,2.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.414468,66683e8865601d05fa511e02,Web Lead,0,0,-1.0,-1.0
7,6a023f6398ffe523164caa45,1,2026-05-11 20:43:16.163,2026-05

In [832]:
_callsDF_3m=_callsDF_6m[_callsDF_6m['lead'].isin(df_master_episode_dataset['lead'].tolist())]

In [834]:
_callsDF_3m.columns

Index(['_id', 'lead', 'createdAt', 'direction', 'status', 'duration'], dtype='str')

In [835]:
df_master_episode_dataset1=df_master_episode_dataset.copy()

In [836]:
# Per-episode call stats from `_leadsCallDF` within [episode_start_time, episode_end_time]

def _col(df, *names):
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n in df.columns:
            return n
        if n.lower() in lower:
            return lower[n.lower()]
    raise KeyError(f"None of {names} found in {list(df.columns)}")

_calls = _callsDF_3m.copy()
_calls["lead"] = _calls[_col(_calls, "lead")].astype(str).str.strip()
_calls["call_time"] = pd.to_datetime(_calls[_col(_calls, "createdAt", "createdAt")], errors="coerce")
_calls["call_status"] = _calls[_col(_calls, "status", "status")].astype(str).str.strip().str.lower()
_calls["call_direction"] = _calls[_col(_calls, "direction")].astype(str).str.strip().str.lower()
_calls["call_duration"] = pd.to_numeric(
    _calls[_col(_calls, "duration", "duration")], errors="coerce"
).fillna(0.0)

_eps = df_master_episode_dataset1.copy()
_eps["lead"] = _eps["lead"].astype(str).str.strip()
_eps["episode_start_time"] = pd.to_datetime(_eps["episode_start_time"], errors="coerce")
_eps["episode_end_time"] = pd.to_datetime(_eps["episode_end_time"], errors="coerce")

_ep_keys = ["lead", "episode_number", "episode_start_time", "episode_end_time"]
if "episode_id" in _eps.columns:
    _ep_keys.append("episode_id")

_merged = _calls.merge(_eps[_ep_keys], on="lead", how="inner")
_in_episode = _merged[
    _merged["call_time"].notna()
    & _merged["episode_start_time"].notna()
    & _merged["episode_end_time"].notna()
    & (_merged["call_time"] >= _merged["episode_start_time"])
    & (_merged["call_time"] <= _merged["episode_end_time"])
].copy()

_in_episode["is_answered"] = _in_episode["call_status"].eq("answered")
_in_episode["is_inbound"] = _in_episode["call_direction"].eq("inbound")
_in_episode["is_outbound"] = _in_episode["call_direction"].eq("outbound")
_in_episode["is_agent_missed"] = _in_episode["call_status"].eq("agent-missed")
_in_episode["is_customer_missed"] = _in_episode["call_status"].eq("customer-missed")
_in_episode["is_attempted"] = _in_episode["call_status"].isin(
    ["answered", "customer-missed"]
)
# duration only counted for answered calls
_in_episode["answered_duration"] = np.where(
    _in_episode["is_answered"], _in_episode["call_duration"], 0.0
)

_stat_cols = [
    "total_answered_calls",
    "total_inbound_calls",
    "total_outbound_calls",
    "total_agent_missed_calls",
    "total_customer_missed_calls",
    "total_attempted_calls",
    "total_call_duration",
]

_episode_call_stats = (
    _in_episode.groupby(["lead", "episode_number"], as_index=False)
    .agg(
        total_answered_calls=("is_answered", "sum"),
        total_inbound_calls=("is_inbound", "sum"),
        total_outbound_calls=("is_outbound", "sum"),
        total_agent_missed_calls=("is_agent_missed", "sum"),
        total_customer_missed_calls=("is_customer_missed", "sum"),
        total_attempted_calls=("is_attempted", "sum"),
        total_call_duration=("answered_duration", "sum"),
    )
)

# Drop prior stats so re-runs don't create _x/_y columns
df_episodes = df_episodes.drop(
    columns=[c for c in _stat_cols if c in df_episodes.columns]
    + [c for c in df_episodes.columns if c.endswith(("_x", "_y")) and any(
        c.startswith(s) for s in _stat_cols
    )],
    errors="ignore",
)

df_episodes = df_episodes.merge(
    _episode_call_stats, on=["lead", "episode_number"], how="left"
)
for _c in _stat_cols:
    df_episodes[_c] = df_episodes[_c].fillna(0)

print(
    "episodes with any answered calls:",
    (df_episodes["total_answered_calls"] > 0).sum(),
)
# df_episodes[
#     [
#         "lead",
#         "episode_number",
#         "episode_start_time",
#         "episode_end_time",
#         "total_answered_calls",
#         "total_inbound_calls",
#         "total_outbound_calls",
#         "total_call_duration",
#     ]
# ].head(10)

episodes with any answered calls: 54336


In [843]:
import time
import pandas as pd
import numpy as np

print("🚀 Starting Per-Episode Call Stats Calculation...")
t0 = time.time()

# 1. Resolve source calls DataFrame dynamically (_callsDF_6m, _callsDF_3m, or _callsDF)
calls_src = (
    _callsDF_3m 
   
)

if calls_src is None:
    raise NameError("❌ Error: No call records DataFrame found (_callsDF_6m, _callsDF_3m, or _callsDF).")

# Helper function to resolve column names robustly
def _col(df, *names):
    lower_map = {c.lower(): c for c in df.columns}
    for n in names:
        if n in df.columns:
            return n
        if n.lower() in lower_map:
            return lower_map[n.lower()]
    raise KeyError(f"None of {names} found in {list(df.columns)}")

# 2. Prepare clean calls dataset
_calls = calls_src.copy()
_calls["lead"] = _calls[_col(_calls, "lead")].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
_calls["call_time"] = pd.to_datetime(_calls[_col(_calls, "createdAt", "createdat")], errors="coerce")
_calls["call_status"] = _calls[_col(_calls, "status", "callstatus")].astype(str).str.strip().str.lower()
_calls["call_direction"] = _calls[_col(_calls, "direction")].astype(str).str.strip().str.lower()
_calls["call_duration"] = pd.to_numeric(_calls[_col(_calls, "duration", "callduration")], errors="coerce").fillna(0.0)

# 3. Prepare target master episode dataset
_eps = df_master_episode_dataset.copy()
_eps["lead"] = _eps["lead"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
_eps["episode_start_time"] = pd.to_datetime(_eps["episode_start_time"], errors="coerce")
_eps["episode_end_time"] = pd.to_datetime(_eps["episode_end_time"], errors="coerce")

_ep_keys = ["lead", "episode_number", "episode_start_time", "episode_end_time"]
if "episode_id" in _eps.columns:
    _ep_keys.append("episode_id")

# 4. Time-window merge: episode_start_time <= call_time <= episode_end_time
_merged = _calls.merge(_eps[_ep_keys], on="lead", how="inner")
_in_episode = _merged[
    _merged["call_time"].notna()
    & _merged["episode_start_time"].notna()
    & _merged["episode_end_time"].notna()
    & (_merged["call_time"] >= _merged["episode_start_time"])
    & (_merged["call_time"] <= _merged["episode_end_time"])
].copy()

# 5. Apply Call Classifications
_in_episode["is_answered"] = _in_episode["call_status"].eq("answered")
_in_episode["is_inbound"] = _in_episode["call_direction"].eq("inbound")
_in_episode["is_outbound"] = _in_episode["call_direction"].eq("outbound")
_in_episode["is_agent_missed"] = _in_episode["call_status"].eq("agent-missed")
_in_episode["is_customer_missed"] = _in_episode["call_status"].eq("customer-missed")
_in_episode["is_attempted"] = _in_episode["call_status"].isin(["answered", "customer-missed"])

# Duration only counted for answered calls
_in_episode["answered_duration"] = np.where(_in_episode["is_answered"], _in_episode["call_duration"], 0.0)

_stat_cols = [
    "total_answered_calls",
    "total_inbound_calls",
    "total_outbound_calls",
    "total_agent_missed_calls",
    "total_customer_missed_calls",
    "total_attempted_calls",
    "total_call_duration",
]

# 6. Aggregate call stats per (lead, episode_number)
_episode_call_stats = (
    _in_episode.groupby(["lead", "episode_number"], as_index=False)
    .agg(
        total_answered_calls=("is_answered", "sum"),
        total_inbound_calls=("is_inbound", "sum"),
        total_outbound_calls=("is_outbound", "sum"),
        total_agent_missed_calls=("is_agent_missed", "sum"),
        total_customer_missed_calls=("is_customer_missed", "sum"),
        total_attempted_calls=("is_attempted", "sum"),
        total_call_duration=("answered_duration", "sum"),
    )
)

# 7. Drop prior stats to avoid _x/_y duplicate columns on re-runs
df_master_episode_dataset = df_master_episode_dataset.drop(
    columns=[c for c in _stat_cols if c in df_master_episode_dataset.columns]
    + [c for c in df_master_episode_dataset.columns if c.endswith(("_x", "_y")) and any(c.startswith(s) for s in _stat_cols)],
    errors="ignore",
)

# Clean merge keys for exact matching
df_master_episode_dataset["lead_clean"] = df_master_episode_dataset["lead"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
_episode_call_stats["lead_clean"] = _episode_call_stats["lead"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Merge onto df_master_episode_dataset
df_master_episode_dataset = df_master_episode_dataset.merge(
    _episode_call_stats.drop(columns=["lead"]), 
    on=["lead_clean", "episode_number"], 
    how="left"
).drop(columns=["lead_clean"], errors="ignore")

# Fill 0 for episodes with no call records
for _c in _stat_cols:
    if _c != "total_call_duration":
        df_master_episode_dataset[_c] = df_master_episode_dataset[_c].fillna(0).astype(int)
    else:
        df_master_episode_dataset[_c] = df_master_episode_dataset[_c].fillna(0.0)

t1 = time.time()

print("=" * 85)
print(f"🎉 SUCCESS! CALL STATS MERGED INTO `df_master_episode_dataset` ({t1 - t0:.2f}s)!")
print("=" * 85)
print("Episodes with answered calls (>0):", (df_master_episode_dataset["total_answered_calls"] > 0).sum())
print("Episodes with inbound calls  (>0):", (df_master_episode_dataset["total_inbound_calls"] > 0).sum())
print("=" * 85)
print("DataFrame Columns:", df_master_episode_dataset.columns.tolist())
df_master_episode_dataset[
    [
        "lead",
        "episode_number",
        "episode_start_time",
        "episode_end_time",
        "total_answered_calls",
        "total_inbound_calls",
        "total_outbound_calls",
        "total_call_duration",
    ]
].head(10)


🚀 Starting Per-Episode Call Stats Calculation...
🎉 SUCCESS! CALL STATS MERGED INTO `df_master_episode_dataset` (1.53s)!
Episodes with answered calls (>0): 54336
Episodes with inbound calls  (>0): 11549
DataFrame Columns: ['lead', 'episode_number', 'episode_start_time', 'episode_end_time', 'start_stage', 'end_stage', 'total_stage_movements', 'episode_id', 'is_reactivated_episode', 'masterLeadId', 'bornSource', 'source', 'secondarySource', 'project', 'status', 'lead_createdAt', 'globalId', 'leadId', 'visitorId', 'total_web_timespent', 'active_session_count', 'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent', 'amenities_timespent', 'media_timespent', 'specifications_timespent', 'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded', 'is_price_sheet_viewed', 'is_cost_sheet_clicked', 'total_session_count', 'resubmission_count_in_episode', 'is_resubmitted_in_episode', 'did_reach_site_visit_in_episode', 'episode_stage_movement_count', 'stage_be

,lead,episode_number,episode_start_time,episode_end_time,total_answered_calls,total_inbound_calls,total_outbound_calls,total_call_duration
0,6a023a53d90c4857b766189c,1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,0,0,10,0.000
1,6a023b888b2d7b451592811f,1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,1,0,1,46.608
2,6a023bb0f858b6e164691c23,1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,4,0,6,661.214
3,6a023c83ddaa0af3065f4e91,1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,1,0,2,109.904
4,6a023d25d90c4857b7661b5d,1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,1,0,6,11.000
5,6a023d3d8b2d7b45159282d0,1,2026-05-11 20:34:06.054,2026-05-12 04:58:20.588,1,0,1,20.000
6,6a023e02f858b6e164691e26,1,2026-05-11 20:37:24.791,2026-05-12 06:34:14.861,1,0,1,31.000
7,6a023f6398ffe523164caa45,1,2026-05-11 20:43:16.163,2026-05-12 07:47:43.976,1,0,2,18.000
8,6a023ffef7f146ffde9fe840,1,2026-05-11 20:45:50.862,2026-05-17 13:14:21.269,0,0,6,0.000
9,6a023ffef7f146ffde9fe840,2,2026-05-17 13:14:21.270,2026-05-23 05:06:00.495,0,0,4,0.000


In [848]:
df_master_episode_dataset.to_excel('FinalData.xlsx')

In [847]:
df_master_episode_dataset['pickup_ratio'] = (df_master_episode_dataset['total_answered_calls'] / df_master_episode_dataset['total_attempted_calls']) * 100

In [849]:
import pandas as pd
import numpy as np

print("=" * 110)
print("🚀 SMART 2-TIERED CONDITIONAL ANALYSIS: REACTIVATED vs NON-REACTIVATED EPISODES")
print("=" * 110)

df_analysis = df_master_episode_dataset.copy()

react_mask = df_analysis['is_reactivated_episode'] == 1
non_react_mask = df_analysis['is_reactivated_episode'] == 0

n_react = react_mask.sum()
n_non_react = non_react_mask.sum()

print(f"  • Non-Reactivated Episodes (0): {n_non_react:,}")
print(f"  • Reactivated Episodes     (1): {n_react:,}")
print("=" * 110)

num_metrics = [
    'total_web_timespent', 'total_session_count', 'active_session_count', 'avg_web_activity',
    'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent',
    'amenities_timespent', 'media_timespent', 'specifications_timespent'
]

flag_metrics = [
    'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded',
    'is_price_sheet_viewed', 'is_cost_sheet_clicked'
]

results = []

# 1. Continuous Metrics (Tier 1: % Active > 0 | Tier 2: Mean & Median when > 0)
for m in num_metrics:
    if m in df_analysis.columns:
        s_non = df_analysis.loc[non_react_mask, m]
        s_rec = df_analysis.loc[react_mask, m]
        
        # Tier 1: Participation Rate (% > 0)
        part_non = (s_non > 0).mean() * 100
        part_rec = (s_rec > 0).mean() * 100
        part_diff = part_rec - part_non
        
        # Tier 2: Conditional Mean & Median (Only on > 0)
        s_non_act = s_non[s_non > 0]
        s_rec_act = s_rec[s_rec > 0]
        
        c_mean_non = s_non_act.mean() if len(s_non_act) > 0 else 0.0
        c_mean_rec = s_rec_act.mean() if len(s_rec_act) > 0 else 0.0
        c_mean_diff = ((c_mean_rec - c_mean_non) / (c_mean_non + 1e-5)) * 100
        
        c_med_non = s_non_act.median() if len(s_non_act) > 0 else 0.0
        c_med_rec = s_rec_act.median() if len(s_rec_act) > 0 else 0.0
        
        results.append({
            'Metric Name': m,
            'Non-React (% Active)': f"{part_non:.1f}%",
            'React (% Active)': f"{part_rec:.1f}%",
            'Active Rate Diff': f"{part_diff:+.1f}% pts",
            'Non-React Cond. Avg': f"{c_mean_non:.1f}",
            'React Cond. Avg': f"{c_mean_rec:.1f}",
            'Cond. Avg Uplift': f"{c_mean_diff:+.1f}%",
            'Non-React Cond. Med': f"{c_med_non:.1f}",
            'React Cond. Med': f"{c_med_rec:.1f}"
        })

# 2. Action Click Boolean Flags (Tier 1 % Adoption)
for f in flag_metrics:
    if f in df_analysis.columns:
        p_non = df_analysis.loc[non_react_mask, f].mean() * 100
        p_rec = df_analysis.loc[react_mask, f].mean() * 100
        diff_pts = p_rec - p_non
        
        results.append({
            'Metric Name': f,
            'Non-React (% Active)': f"{p_non:.1f}%",
            'React (% Active)': f"{p_rec:.1f}%",
            'Active Rate Diff': f"{diff_pts:+.1f}% pts",
            'Non-React Cond. Avg': "N/A (Flag)",
            'React Cond. Avg': "N/A (Flag)",
            'Cond. Avg Uplift': "N/A",
            'Non-React Cond. Med': "N/A",
            'React Cond. Med': "N/A"
        })

df_smart_report = pd.DataFrame(results)
print(df_smart_report.to_string(index=False))


🚀 SMART 2-TIERED CONDITIONAL ANALYSIS: REACTIVATED vs NON-REACTIVATED EPISODES
  • Non-Reactivated Episodes (0): 105,796
  • Reactivated Episodes     (1): 5,942
               Metric Name Non-React (% Active) React (% Active) Active Rate Diff Non-React Cond. Avg React Cond. Avg Cond. Avg Uplift Non-React Cond. Med React Cond. Med
       total_web_timespent                 3.3%             3.8%        +0.5% pts               198.1           571.2          +188.4%                30.3            63.6
       total_session_count                 5.1%             5.0%        -0.1% pts                 1.8             2.1           +12.8%                 1.0             1.0
      active_session_count                 3.3%             3.8%        +0.5% pts                 1.5             1.8           +16.7%                 1.0             1.0
            home_timespent                 2.6%             2.7%        +0.2% pts               156.3           369.9          +136.6%                21.8 

In [873]:
import pandas as pd
import numpy as np

print("=" * 115)
print("🔍 OUTLIER DIAGNOSTIC & PERCENTILE BREAKDOWN (ACTIVE VALUES ONLY > 0)")
print("=" * 115)

df_analysis = df_master_episode_dataset.copy()

react_mask = df_analysis['is_reactivated_episode'] == 1
non_react_mask = df_analysis['is_reactivated_episode'] == 0

metrics = [
    'total_web_timespent',  'avg_web_activity',
    'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent',
    'amenities_timespent', 'media_timespent', 'specifications_timespent'
]

rows = []

for m in metrics:
    if m in df_analysis.columns:
        s_non = df_analysis.loc[non_react_mask, m]
        s_rec = df_analysis.loc[react_mask, m]
        
        s_non_act = s_non[s_non > 0]
        s_rec_act = s_rec[s_rec > 0]
        
        if len(s_non_act) > 0 and len(s_rec_act) > 0:
            # Raw Means
            raw_mean_non = s_non_act.mean()
            raw_mean_rec = s_rec_act.mean()
            
            # Percentiles
            p50_non, p50_rec = s_non_act.median(), s_rec_act.median()
            p75_non, p75_rec = s_non_act.quantile(0.75), s_rec_act.quantile(0.75)
            p90_non, p90_rec = s_non_act.quantile(0.90), s_rec_act.quantile(0.90)
            
            # P95 Capped Trimmed Mean
            p95_cap = s_non_act.quantile(0.95)
            c_mean_non_p95 = s_non_act.clip(upper=p95_cap).mean()
            c_mean_rec_p95 = s_rec_act.clip(upper=p95_cap).mean()
            c_p95_uplift = ((c_mean_rec_p95 - c_mean_non_p95) / (c_mean_non_p95 + 1e-5)) * 100
            
            # Outliers > 3600s (1 hour)
            out_non = (s_non_act > 3600).sum()
            out_rec = (s_rec_act > 3600).sum()
            
            rows.append({
                'Metric Name': m,
                'Non-React Raw Avg': f"{raw_mean_non:.1f}s",
                'React Raw Avg': f"{raw_mean_rec:.1f}s",
                'Non-React P50 (Med)': f"{p50_non:.1f}s",
                'React P50 (Med)': f"{p50_rec:.1f}s",
                'Non-React P75': f"{p75_non:.1f}s",
                'React P75': f"{p75_rec:.1f}s",
                'Non-React P90': f"{p90_non:.1f}s",
                'React P90': f"{p90_rec:.1f}s",
                'Non-React P95-Capped Avg': f"{c_mean_non_p95:.1f}s",
                'React P95-Capped Avg': f"{c_mean_rec_p95:.1f}s",
                'Capped Uplift (%)': f"{c_p95_uplift:+.1f}%",
                'Outliers >1hr (Non/React)': f"{out_non} / {out_rec}"
            })

df_outlier_report = pd.DataFrame(rows)
print(df_outlier_report.to_string(index=False))


🔍 OUTLIER DIAGNOSTIC & PERCENTILE BREAKDOWN (ACTIVE VALUES ONLY > 0)
             Metric Name Non-React Raw Avg React Raw Avg Non-React P50 (Med) React P50 (Med) Non-React P75 React P75 Non-React P90 React P90 Non-React P95-Capped Avg React P95-Capped Avg Capped Uplift (%) Outliers >1hr (Non/React)
     total_web_timespent            198.1s        571.2s               30.3s           63.6s         98.7s    197.8s        250.9s    447.8s                    80.4s               125.7s            +56.3%                    12 / 5
          home_timespent            156.3s        369.9s               21.8s           31.0s         60.1s     96.9s        147.9s    234.3s                    50.1s                69.0s            +37.7%                     4 / 2
         price_timespent             98.6s        170.3s               29.7s           51.3s         95.5s    148.5s        188.7s    430.1s                    68.4s                99.9s            +46.0%                     2 / 0
       

In [ ]:
# import pandas as pd
# import numpy as np

# print("=" * 110)
# print("🚀 EPISODE STAGE & TRANSITION PATH ANALYSIS (REACTIVATED vs NON-REACTIVATED)")
# print("=" * 110)

# df_analysis = df_master_episode_dataset.copy()

# react_mask = df_analysis['is_reactivated_episode'] == 1
# non_react_mask = df_analysis['is_reactivated_episode'] == 0

# n_react = react_mask.sum()
# n_non_react = non_react_mask.sum()

# print(f"  • Non-Reactivated Episodes (0): {n_non_react:,}")
# print(f"  • Reactivated Episodes     (1): {n_react:,}")
# print("=" * 110)

# # ==============================================================================
# # PART 1: STAGE & RESUBMISSION METRICS COMPARISON
# # ==============================================================================
# stage_num_metrics = ['resubmission_count_in_episode', 'episode_stage_movement_count', 'episode_duration_days']
# stage_flag_metrics = ['is_resubmitted_in_episode', 'did_reach_site_visit_in_episode']

# results = []

# for m in stage_num_metrics:
#     if m in df_analysis.columns:
#         s_non = df_analysis.loc[non_react_mask, m]
#         s_rec = df_analysis.loc[react_mask, m]
        
#         m_non, m_rec = s_non.mean(), s_rec.mean()
#         med_non, med_rec = s_non.median(), s_rec.median()
#         diff_pct = ((m_rec - m_non) / (m_non + 1e-5)) * 100
        
#         results.append({
#             'Metric Name': m,
#             'Non-React (Avg)': f"{m_non:.2f}",
#             'React (Avg)': f"{m_rec:.2f}",
#             'Non-React (Median)': f"{med_non:.2f}",
#             'React (Median)': f"{med_rec:.2f}",
#             'Uplift / Diff (%)': f"{diff_pct:+.2f}%"
#         })

# for f in stage_flag_metrics:
#     if f in df_analysis.columns:
#         p_non = df_analysis.loc[non_react_mask, f].mean() * 100
#         p_rec = df_analysis.loc[react_mask, f].mean() * 100
#         diff_pts = p_rec - p_non
        
#         results.append({
#             'Metric Name': f,
#             'Non-React (Avg)': f"{p_non:.2f}%",
#             'React (Avg)': f"{p_rec:.2f}%",
#             'Non-React (Median)': "N/A",
#             'React (Median)': "N/A",
#             'Uplift / Diff (%)': f"{diff_pts:+.2f}% pts"
#         })

# print("\n📊 PART 1: STAGE & RESUBMISSION SUMMARY METRICS")
# print("-" * 110)
# print(pd.DataFrame(results).to_string(index=False))

# ==============================================================================
# PART 2: STAGE TRANSITION PATH ANALYSIS (`start_stage -> stage_before_ni`)
# ==============================================================================
print("\n" + "=" * 110)
print("📊 PART 2: TOP STAGE TRANSITION PATHS (`start_stage -> stage_before_ni`)")
print("=" * 110)

if 'start_stage' in df_analysis.columns and 'stage_before_ni' in df_analysis.columns:
    df_analysis['stage_transition_path'] = (
        df_analysis['start_stage'].astype(str).str.strip() + " ➔ " + df_analysis['stage_before_ni'].astype(str).str.strip()
    )
    
    # Calculate percentage distribution per cohort
    ct_path = pd.crosstab(df_analysis['stage_transition_path'], df_analysis['is_reactivated_episode'], normalize='columns') * 100
    ct_path.columns = ['Non-Reactivated %', 'Reactivated %']
    ct_path['Diff (% pts)'] = ct_path['Reactivated %'] - ct_path['Non-Reactivated %']
    
    # Sort by Reactivated %
    ct_path = ct_path.sort_values(by='Reactivated %', ascending=False).head(15).reset_index()
    
    # Format strings
    ct_path['Non-Reactivated %'] = ct_path['Non-Reactivated %'].map('{:.2f}%'.format)
    ct_path['Reactivated %'] = ct_path['Reactivated %'].map('{:.2f}%'.format)
    ct_path['Diff (% pts)'] = ct_path['Diff (% pts)'].map('{:+.2f}% pts'.format)
    
    print(ct_path.to_string(index=False))

# ==============================================================================
# PART 3: TOP EPISODE SOURCES COMPARISON (`episode_source_name`)
# ==============================================================================
print("\n" + "=" * 110)
print("📊 PART 3: TOP EPISODE SOURCES DISTRIBUTION (`episode_source_name`)")
print("=" * 110)

if 'episode_source_name' in df_analysis.columns:
    ct_src = pd.crosstab(df_analysis['episode_source_name'], df_analysis['is_reactivated_episode'], normalize='columns') * 100
    ct_src.columns = ['Non-Reactivated %', 'Reactivated %']
    ct_src['Diff (% pts)'] = ct_src['Reactivated %'] - ct_src['Non-Reactivated %']
    
    ct_src = ct_src.sort_values(by='Reactivated %', ascending=False).head(15).reset_index()
    
    ct_src['Non-Reactivated %'] = ct_src['Non-Reactivated %'].map('{:.2f}%'.format)
    ct_src['Reactivated %'] = ct_src['Reactivated %'].map('{:.2f}%'.format)
    ct_src['Diff (% pts)'] = ct_src['Diff (% pts)'].map('{:+.2f}% pts'.format)
    
    print(ct_src.to_string(index=False))


🚀 EPISODE STAGE & TRANSITION PATH ANALYSIS (REACTIVATED vs NON-REACTIVATED)
  • Non-Reactivated Episodes (0): 105,796
  • Reactivated Episodes     (1): 5,942

📊 PART 1: STAGE & RESUBMISSION SUMMARY METRICS
--------------------------------------------------------------------------------------------------------------
                    Metric Name Non-React (Avg) React (Avg) Non-React (Median) React (Median) Uplift / Diff (%)
  resubmission_count_in_episode            0.08        0.21               0.00           0.00          +155.33%
   episode_stage_movement_count            2.13        2.65               2.00           3.00           +24.61%
          episode_duration_days            4.51        5.74               0.59           3.87           +27.36%
      is_resubmitted_in_episode           5.46%      13.43%                N/A            N/A        +7.97% pts
did_reach_site_visit_in_episode           1.63%       4.01%                N/A            N/A        +2.37% pts

📊 PART 2: 

In [875]:
import pandas as pd
import numpy as np

print("=" * 115)
print("🚀 STAGE & RESUBMISSION METRICS: CORRECTED 2-TIERED CONDITIONAL ANALYSIS")
print("=" * 115)

df_analysis = df_master_episode_dataset.copy()

react_mask = df_analysis['is_reactivated_episode'] == 1
non_react_mask = df_analysis['is_reactivated_episode'] == 0

stage_num_metrics = ['resubmission_count_in_episode', 'episode_stage_movement_count', 'episode_duration_days']
stage_flag_metrics = ['is_resubmitted_in_episode', 'did_reach_site_visit_in_episode']

results = []

for m in stage_num_metrics:
    if m in df_analysis.columns:
        s_non = df_analysis.loc[non_react_mask, m]
        s_rec = df_analysis.loc[react_mask, m]
        
        # For duration, include all valid episodes (>= 0); for counts, filter (> 0)
        if m == "episode_duration_days":
            part_non, part_rec = 100.0, 100.0
            s_non_act, s_rec_act = s_non[s_non >= 0], s_rec[s_rec >= 0]
        else:
            part_non, part_rec = (s_non > 0).mean() * 100, (s_rec > 0).mean() * 100
            s_non_act, s_rec_act = s_non[s_non > 0], s_rec[s_rec > 0]
            
        mean_non = s_non_act.mean() if len(s_non_act) > 0 else 0.0
        mean_rec = s_rec_act.mean() if len(s_rec_act) > 0 else 0.0
        med_non = s_non_act.median() if len(s_non_act) > 0 else 0.0
        med_rec = s_rec_act.median() if len(s_rec_act) > 0 else 0.0
        
        uplift = ((mean_rec - mean_non) / (mean_non + 1e-5)) * 100
        unit = " days" if m == "episode_duration_days" else ""
        
        results.append({
            'Metric Name': m,
            'Non-React (% >0)': f"{part_non:.2f}%",
            'React (% >0)': f"{part_rec:.2f}%",
            'Active Rate Diff': f"{part_rec - part_non:+.2f}% pts",
            'Non-React Cond Avg': f"{mean_non:.2f}{unit}",
            'React Cond Avg': f"{mean_rec:.2f}{unit}",
            'Cond Avg Uplift': f"{uplift:+.2f}%",
            'Non-React Cond Med': f"{med_non:.2f}{unit}",
            'React Cond Med': f"{med_rec:.2f}{unit}"
        })

for f in stage_flag_metrics:
    if f in df_analysis.columns:
        p_non = df_analysis.loc[non_react_mask, f].mean() * 100
        p_rec = df_analysis.loc[react_mask, f].mean() * 100
        diff_pts = p_rec - p_non
        
        results.append({
            'Metric Name': f,
            'Non-React (% >0)': f"{p_non:.2f}%",
            'React (% >0)': f"{p_rec:.2f}%",
            'Active Rate Diff': f"{diff_pts:+.2f}% pts",
            'Non-React Cond Avg': "N/A (Flag)",
            'React Cond Avg': "N/A (Flag)",
            'Cond Avg Uplift': "N/A",
            'Non-React Cond Med': "N/A",
            'React Cond Med': "N/A"
        })

df_stage_2tier_report = pd.DataFrame(results)
print(df_stage_2tier_report.to_string(index=False))


🚀 STAGE & RESUBMISSION METRICS: CORRECTED 2-TIERED CONDITIONAL ANALYSIS
                    Metric Name Non-React (% >0) React (% >0) Active Rate Diff Non-React Cond Avg React Cond Avg Cond Avg Uplift Non-React Cond Med React Cond Med
  resubmission_count_in_episode            5.46%       13.43%       +7.97% pts               1.50           1.56          +3.86%               1.00           1.00
   episode_stage_movement_count          100.00%      100.00%       +0.00% pts               2.13           2.65         +24.61%               2.00           3.00
          episode_duration_days          100.00%      100.00%       +0.00% pts          4.51 days      5.74 days         +27.36%          0.59 days      3.87 days
      is_resubmitted_in_episode            5.46%       13.43%       +7.97% pts         N/A (Flag)     N/A (Flag)             N/A                N/A            N/A
did_reach_site_visit_in_episode            1.63%        4.01%       +2.37% pts         N/A (Flag)     N/A (Flag) 

In [876]:
# import pandas as pd
# import numpy as np

# print("=" * 95)
# print("🔍 CONDITIONAL BREAKDOWN FOR `resubmission_count_in_episode` (WHEN > 0)")
# print("=" * 95)

# df_analysis = df_master_episode_dataset.copy()

# react_mask = df_analysis['is_reactivated_episode'] == 1
# non_react_mask = df_analysis['is_reactivated_episode'] == 0

# s_non = df_analysis.loc[non_react_mask, 'resubmission_count_in_episode']
# s_rec = df_analysis.loc[react_mask, 'resubmission_count_in_episode']

# # Tier 1: Participation Rate (% > 0)
# part_non = (s_non > 0).mean() * 100
# part_rec = (s_rec > 0).mean() * 100
# part_diff = part_rec - part_non

# # Tier 2: Conditional Intensity (Only on > 0)
# s_non_act = s_non[s_non > 0]
# s_rec_act = s_rec[s_rec > 0]

# cond_mean_non = s_non_act.mean() if len(s_non_act) > 0 else 0.0
# cond_mean_rec = s_rec_act.mean() if len(s_rec_act) > 0 else 0.0
# cond_mean_diff = ((cond_mean_rec - cond_mean_non) / (cond_mean_non + 1e-5)) * 100

# cond_med_non = s_non_act.median() if len(s_non_act) > 0 else 0.0
# cond_med_rec = s_rec_act.median() if len(s_rec_act) > 0 else 0.0

# res_df = pd.DataFrame([{
#     'Metric Name': 'resubmission_count_in_episode',
#     'Non-React (% >0)': f"{part_non:.2f}%",
#     'React (% >0)': f"{part_rec:.2f}%",
#     'Active Rate Diff': f"{part_diff:+.2f}% pts",
#     'Non-React Cond Avg': f"{cond_mean_non:.2f}",
#     'React Cond Avg': f"{cond_mean_rec:.2f}",
#     'Cond Avg Uplift': f"{cond_mean_diff:+.2f}%",
#     'Non-React Cond Med': f"{cond_med_non:.2f}",
#     'React Cond Med': f"{cond_med_rec:.2f}"
# }])

# print(res_df.to_string(index=False))


In [853]:
import pandas as pd
import numpy as np

print("=" * 115)
print("🚀 FINAL BATCH ANALYSIS: WHATSAPP INBOUND METRICS & AVG WEB ACTIVITY")
print("=" * 115)

df_analysis = df_master_episode_dataset.copy()

react_mask = df_analysis['is_reactivated_episode'] == 1
non_react_mask = df_analysis['is_reactivated_episode'] == 0

n_react = react_mask.sum()
n_non_react = non_react_mask.sum()

print(f"  • Non-Reactivated Episodes (0): {n_non_react:,}")
print(f"  • Reactivated Episodes     (1): {n_react:,}")
print("=" * 115)

# ------------------------------------------------------------------------------
# PART 1: WHATSAPP INBOUND ADOPTION RATE (has_whatsapp_inbound_in_episode)
# ------------------------------------------------------------------------------
p_non_wa = df_analysis.loc[non_react_mask, 'has_whatsapp_inbound_in_episode'].mean() * 100
p_rec_wa = df_analysis.loc[react_mask, 'has_whatsapp_inbound_in_episode'].mean() * 100
wa_diff = p_rec_wa - p_non_wa

print(f"\n📊 1. WHATSAPP INBOUND ADOPTION RATE (`has_whatsapp_inbound_in_episode`):")
print("-" * 115)
print(f"  • Non-Reactivated Episodes: {p_non_wa:.2f}%")
print(f"  • Reactivated Episodes:     {p_rec_wa:.2f}%")
print(f"  • Difference (% pts):       {wa_diff:+.2f}% pts")

# ------------------------------------------------------------------------------
# PART 2: CONDITIONAL ANALYSIS FOR COUNTS & SPEED (WHEN > 0 OR VALID DAYS >= 0)
# ------------------------------------------------------------------------------
results = []

# A. WhatsApp Inbound Message Count (when count > 0)
m = 'episode_whatsapp_inbound_count'
if m in df_analysis.columns:
    s_non = df_analysis.loc[non_react_mask, m]
    s_rec = df_analysis.loc[react_mask, m]
    
    part_non, part_rec = (s_non > 0).mean() * 100, (s_rec > 0).mean() * 100
    
    s_non_act, s_rec_act = s_non[s_non > 0], s_rec[s_rec > 0]
    
    mean_non = s_non_act.mean() if len(s_non_act) > 0 else 0.0
    mean_rec = s_rec_act.mean() if len(s_rec_act) > 0 else 0.0
    med_non = s_non_act.median() if len(s_non_act) > 0 else 0.0
    med_rec = s_rec_act.median() if len(s_rec_act) > 0 else 0.0
    
    results.append({
        'Metric Name': m,
        'Non-React (% Active)': f"{part_non:.2f}%",
        'React (% Active)': f"{part_rec:.2f}%",
        'Non-React Cond Avg': f"{mean_non:.2f}",
        'React Cond Avg': f"{mean_rec:.2f}",
        'Cond Avg Uplift': f"{((mean_rec - mean_non)/(mean_non+1e-5))*100:+.2f}%",
        'Non-React Cond Med': f"{med_non:.2f}",
        'React Cond Med': f"{med_rec:.2f}"
    })

# B. Average Web Activity (when > 0)
m = 'avg_web_activity'
if m in df_analysis.columns:
    s_non = df_analysis.loc[non_react_mask, m]
    s_rec = df_analysis.loc[react_mask, m]
    
    part_non, part_rec = (s_non > 0).mean() * 100, (s_rec > 0).mean() * 100
    s_non_act, s_rec_act = s_non[s_non > 0], s_rec[s_rec > 0]
    
    mean_non = s_non_act.mean() if len(s_non_act) > 0 else 0.0
    mean_rec = s_rec_act.mean() if len(s_rec_act) > 0 else 0.0
    med_non = s_non_act.median() if len(s_non_act) > 0 else 0.0
    med_rec = s_rec_act.median() if len(s_rec_act) > 0 else 0.0
    
    results.append({
        'Metric Name': m,
        'Non-React (% Active)': f"{part_non:.2f}%",
        'React (% Active)': f"{part_rec:.2f}%",
        'Non-React Cond Avg': f"{mean_non:.1f}s",
        'React Cond Avg': f"{mean_rec:.1f}s",
        'Cond Avg Uplift': f"{((mean_rec - mean_non)/(mean_non+1e-5))*100:+.2f}%",
        'Non-React Cond Med': f"{med_non:.1f}s",
        'React Cond Med': f"{med_rec:.1f}s"
    })

# C. Days to First & Last WhatsApp Inbound (when >= 0)
day_metrics = ['days_to_first_whatsapp_inbound', 'days_to_last_whatsapp_inbound']
for dm in day_metrics:
    if dm in df_analysis.columns:
        s_non = df_analysis.loc[non_react_mask, dm]
        s_rec = df_analysis.loc[react_mask, dm]
        
        s_non_valid = s_non[s_non >= 0]
        s_rec_valid = s_rec[s_rec >= 0]
        
        mean_non = s_non_valid.mean() if len(s_non_valid) > 0 else 0.0
        mean_rec = s_rec_valid.mean() if len(s_rec_valid) > 0 else 0.0
        med_non = s_non_valid.median() if len(s_non_valid) > 0 else 0.0
        med_rec = s_rec_valid.median() if len(s_rec_valid) > 0 else 0.0
        
        results.append({
            'Metric Name': dm,
            'Non-React (% Active)': f"{(len(s_non_valid)/n_non_react)*100:.2f}%",
            'React (% Active)': f"{(len(s_rec_valid)/n_react)*100:.2f}%",
            'Non-React Cond Avg': f"{mean_non:.2f} days",
            'React Cond Avg': f"{mean_rec:.2f} days",
            'Cond Avg Uplift': f"{((mean_rec - mean_non)/(mean_non+1e-5))*100:+.2f}%",
            'Non-React Cond Med': f"{med_non:.2f} days",
            'React Cond Med': f"{med_rec:.2f} days"
        })

print("\n📊 2. CONDITIONAL ANALYSIS TABLE:")
print("-" * 115)
print(pd.DataFrame(results).to_string(index=False))


🚀 FINAL BATCH ANALYSIS: WHATSAPP INBOUND METRICS & AVG WEB ACTIVITY
  • Non-Reactivated Episodes (0): 105,796
  • Reactivated Episodes     (1): 5,942

📊 1. WHATSAPP INBOUND ADOPTION RATE (`has_whatsapp_inbound_in_episode`):
-------------------------------------------------------------------------------------------------------------------
  • Non-Reactivated Episodes: 1.72%
  • Reactivated Episodes:     1.70%
  • Difference (% pts):       -0.02% pts

📊 2. CONDITIONAL ANALYSIS TABLE:
-------------------------------------------------------------------------------------------------------------------
                   Metric Name Non-React (% Active) React (% Active) Non-React Cond Avg React Cond Avg Cond Avg Uplift Non-React Cond Med React Cond Med
episode_whatsapp_inbound_count                1.72%            1.70%               1.63           1.65          +1.20%               1.00           1.00
days_to_first_whatsapp_inbound                1.72%            1.70%          0.87 days    

In [855]:
import pandas as pd
import numpy as np

print("=" * 115)
print("🚀 CONDITIONAL ANALYSIS: EPISODE CALL METRICS & PICKUP RATIO")
print("=" * 115)

df_analysis = df_master_episode_dataset.copy()

# 1. Compute division-safe pickup_ratio = total_answered_calls / total_attempted_calls
if 'total_attempted_calls' in df_analysis.columns and 'total_answered_calls' in df_analysis.columns:
    df_analysis['pickup_ratio'] = np.where(
        df_analysis['total_attempted_calls'] > 0,
        df_analysis['total_answered_calls'] / df_analysis['total_attempted_calls'],
        0.0
    )

react_mask = df_analysis['is_reactivated_episode'] == 1
non_react_mask = df_analysis['is_reactivated_episode'] == 0

n_react = react_mask.sum()
n_non_react = non_react_mask.sum()

print(f"  • Non-Reactivated Episodes (0): {n_non_react:,}")
print(f"  • Reactivated Episodes     (1): {n_react:,}")
print("=" * 115)

call_metrics = [
    'total_answered_calls', 
    'total_inbound_calls', 
    'total_outbound_calls',
    'total_agent_missed_calls', 
    'total_customer_missed_calls',
    'total_attempted_calls', 
    'total_call_duration', 
    'pickup_ratio'
]

results = []

for m in call_metrics:
    if m in df_analysis.columns:
        s_non = df_analysis.loc[non_react_mask, m]
        s_rec = df_analysis.loc[react_mask, m]
        
        # Tier 1: Participation Rate (% > 0)
        part_non = (s_non > 0).mean() * 100
        part_rec = (s_rec > 0).mean() * 100
        
        # Tier 2: Conditional Mean & Median (Only on > 0)
        s_non_act = s_non[s_non > 0]
        s_rec_act = s_rec[s_rec > 0]
        
        mean_non = s_non_act.mean() if len(s_non_act) > 0 else 0.0
        mean_rec = s_rec_act.mean() if len(s_rec_act) > 0 else 0.0
        med_non = s_non_act.median() if len(s_non_act) > 0 else 0.0
        med_rec = s_rec_act.median() if len(s_rec_act) > 0 else 0.0
        
        uplift = ((mean_rec - mean_non) / (mean_non + 1e-5)) * 100
        
        # Format units cleanly
        unit = "s" if m == "total_call_duration" else ("%" if m == "pickup_ratio" else "")
        
        results.append({
            'Metric Name': m,
            'Non-React (% Active)': f"{part_non:.2f}%",
            'React (% Active)': f"{part_rec:.2f}%",
            'Active Rate Diff': f"{part_rec - part_non:+.2f}% pts",
            'Non-React Cond Avg': f"{mean_non:.2f}{unit}",
            'React Cond Avg': f"{mean_rec:.2f}{unit}",
            'Cond Avg Uplift': f"{uplift:+.2f}%",
            'Non-React Cond Med': f"{med_non:.2f}{unit}",
            'React Cond Med': f"{med_rec:.2f}{unit}"
        })

df_call_report = pd.DataFrame(results)
print(pd.DataFrame(results).to_string(index=False))


🚀 CONDITIONAL ANALYSIS: EPISODE CALL METRICS & PICKUP RATIO
  • Non-Reactivated Episodes (0): 105,796
  • Reactivated Episodes     (1): 5,942
                Metric Name Non-React (% Active) React (% Active) Active Rate Diff Non-React Cond Avg React Cond Avg Cond Avg Uplift Non-React Cond Med React Cond Med
       total_answered_calls               48.13%           57.46%       +9.32% pts               1.74           2.04         +17.13%               1.00           1.00
        total_inbound_calls                9.84%           19.24%       +9.40% pts               1.45           1.77         +21.72%               1.00           1.00
       total_outbound_calls               67.38%           93.50%      +26.12% pts               5.53           6.61         +19.69%               3.00           5.00
   total_agent_missed_calls               42.11%           68.58%      +26.47% pts               3.11           3.33          +7.29%               2.00           3.00
total_customer_missed_c

In [856]:
import time
import pandas as pd
import numpy as np

print("🚀 Building Within-Lead Paired Episode Dataset (Episode x vs Episode x-1)...")
t0 = time.time()

# 1. Clean dataset copy
df_ep = df_master_episode_dataset.copy()
df_ep['lead_clean'] = df_ep['lead'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# 2. Filter for current reactivated episodes (Episode x where x >= 2)
df_curr = df_ep[df_ep['episode_number'] >= 2].copy()
df_curr['prev_episode_number'] = df_curr['episode_number'] - 1

# 3. Target previous episode dataset (Episode x-1)
df_prev = df_ep.copy()

# 4. Merge Episode x with Episode x-1 for the exact same lead
df_paired_episodes = df_curr.merge(
    df_prev,
    left_on=['lead_clean', 'prev_episode_number'],
    right_on=['lead_clean', 'episode_number'],
    suffixes=('_curr', '_prev')
).drop(columns=['lead_clean'], errors='ignore')

# 5. List of numerical metrics to calculate Deltas (Current - Previous)
delta_metrics = [
    'total_web_timespent', 'active_session_count', 'total_session_count', 'avg_web_activity',
    'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent',
    'total_answered_calls', 'total_inbound_calls', 'total_outbound_calls',
    'total_call_duration', 'episode_whatsapp_inbound_count', 'resubmission_count_in_episode',
    'episode_stage_movement_count', 'episode_duration_days'
]

# Calculate Deltas: (Episode x - Episode x-1)
for m in delta_metrics:
    m_curr = f"{m}_curr"
    m_prev = f"{m}_prev"
    if m_curr in df_paired_episodes.columns and m_prev in df_paired_episodes.columns:
        df_paired_episodes[f"delta_{m}"] = df_paired_episodes[m_curr] - df_paired_episodes[m_prev]

t1 = time.time()

print("=" * 85)
print(f"🎉 SUCCESS! PAIRED EPISODE DATASET CREATED ({t1 - t0:.2f} SECONDS)!")
print("=" * 85)
print(f"  • Total Paired Reactivated Episodes (x >= 2): {len(df_paired_episodes):,}")
print(f"  • Unique Leads Represented:                 {df_paired_episodes['lead_curr'].nunique():,}")
print("=" * 85)
df_paired_episodes[['lead_curr', 'episode_number_curr', 'episode_number_prev', 'total_web_timespent_prev', 'total_web_timespent_curr', 'delta_total_web_timespent']].head(10)


🚀 Building Within-Lead Paired Episode Dataset (Episode x vs Episode x-1)...
🎉 SUCCESS! PAIRED EPISODE DATASET CREATED (0.11 SECONDS)!
  • Total Paired Reactivated Episodes (x >= 2): 5,942
  • Unique Leads Represented:                 5,023


,lead_curr,episode_number_curr,episode_number_prev,total_web_timespent_prev,total_web_timespent_curr,delta_total_web_timespent
0,6a023ffef7f146ffde9fe840,2,1,1.155,2009.661,2008.506
1,6a026d151cfdd50b4d63af56,2,1,0.000,0.000,0.000
2,6a026d151cfdd50b4d63af56,3,2,0.000,0.000,0.000
3,6a0278eef7f146ffdea0031f,2,1,0.000,0.000,0.000
4,6a0278eef7f146ffdea0031f,3,2,0.000,0.000,0.000
5,6a027ab598ffe523164cbf27,2,1,0.000,0.000,0.000
6,6a027bc0b1b85af895ce3081,2,1,0.000,0.000,0.000
7,6a0284e6b1b85af895ce3c39,2,1,0.000,0.000,0.000
8,6a0285759a93b65ed4c5bba4,2,1,0.000,0.000,0.000
9,6a028613f0a0af2c8db8e63f,2,1,0.000,0.000,0.000


In [868]:
import pandas as pd
import numpy as np
from scipy import stats

print("=" * 125)
print("🚀 UNDILUTED LONGITUDINAL SCORECARD (ACTIVE PAIRS ONLY: Prev > 0 OR Curr > 0)")
print("=" * 125)

df_paired = df_paired_episodes.copy()
n_total_pairs = len(df_paired)

all_target_metrics = [
    # 1. Web Timespents & Activity
    'total_web_timespent', 'avg_web_activity', 'active_session_count', 'total_session_count',
    'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent',
    'amenities_timespent', 'media_timespent', 'specifications_timespent',
    
    # 2. Action Click Flags
    'is_site_visit_book_clicked', 'is_whatsapp_clicked', 'is_brochure_downloaded',
    'is_price_sheet_viewed', 'is_cost_sheet_clicked',
    
    # 3. Call Metrics
    'total_answered_calls', 'total_inbound_calls', 'total_outbound_calls',
    'total_agent_missed_calls', 'total_customer_missed_calls', 'total_attempted_calls',
    'total_call_duration',
    
    # 4. WhatsApp & Stage Metrics
    'episode_whatsapp_inbound_count', 'resubmission_count_in_episode',
    'episode_stage_movement_count', 'episode_duration_days'
]

scorecard_rows = []

for m in all_target_metrics:
    m_prev_col = f"{m}_prev"
    m_curr_col = f"{m}_curr"
    
    if m_prev_col in df_paired.columns and m_curr_col in df_paired.columns:
        v_prev = pd.to_numeric(df_paired[m_prev_col], errors='coerce').fillna(0.0)
        v_curr = pd.to_numeric(df_paired[m_curr_col], errors='coerce').fillna(0.0)
        
        # ACTIVE PAIRS ONLY MASK (Prev > 0 OR Curr > 0)
        # For duration/days, use valid days >= 0
        if "days" in m:
            active_mask = (v_prev >= 0) | (v_curr >= 0)
        else:
            active_mask = (v_prev > 0) | (v_curr > 0)
            
        v_prev_act = v_prev[active_mask]
        v_curr_act = v_curr[active_mask]
        
        n_active_pairs = len(v_prev_act)
        
        if n_active_pairs > 0:
            delta = v_curr_act - v_prev_act
            
            avg_prev = v_prev_act.mean()
            avg_curr = v_curr_act.mean()
            avg_delta = delta.mean()
            
            pct_uplift = ((avg_curr - avg_prev) / (avg_prev + 1e-5)) * 100
            
            pct_increased = (delta > 0).mean() * 100
            pct_unchanged = (delta == 0).mean() * 100
            pct_decreased = (delta < 0).mean() * 100
            
            # Paired t-test on active pairs
            try:
                if (delta == 0).all():
                    p_val = 1.0
                else:
                    p_val = stats.ttest_rel(v_curr_act, v_prev_act).pvalue
            except Exception:
                p_val = 1.0
                
            sig = "⭐⭐⭐ (p<0.001)" if p_val < 0.001 else ("⭐⭐ (p<0.01)" if p_val < 0.01 else ("⭐ (p<0.05)" if p_val < 0.05 else "NS"))
            
            unit = "s" if "timespent" in m or m in ["total_call_duration", "avg_web_activity"] else (" days" if "days" in m else "")
            
            scorecard_rows.append({
                'Metric Name': m,
                'Active Pairs Count': f"{n_active_pairs:,} ({(n_active_pairs/n_total_pairs)*100:.1f}%)",
                'Prev Ep (x-1)': f"{avg_prev:.2f}{unit}",
                'Curr Ep (x)': f"{avg_curr:.2f}{unit}",
                'Mean Shift (Δ)': f"{avg_delta:+.2f}{unit}",
                'Uplift (%)': f"{pct_uplift:+.2f}%",
                '% Increased': f"{pct_increased:.1f}%",
                '% Unchanged': f"{pct_unchanged:.1f}%",
                '% Decreased': f"{pct_decreased:.1f}%",
                'Significance': sig
            })

df_active_scorecard = pd.DataFrame(scorecard_rows)
print(df_active_scorecard.to_string(index=False))


🚀 UNDILUTED LONGITUDINAL SCORECARD (ACTIVE PAIRS ONLY: Prev > 0 OR Curr > 0)
                   Metric Name Active Pairs Count Prev Ep (x-1) Curr Ep (x) Mean Shift (Δ) Uplift (%) % Increased % Unchanged % Decreased  Significance
           total_web_timespent         335 (5.6%)       387.05s     166.04s       -221.01s    -57.10%       42.4%        0.0%       57.6%            NS
          active_session_count         335 (5.6%)          1.21        0.98          -0.23    -19.06%       38.5%        5.7%       55.8%            NS
           total_session_count         406 (6.8%)          1.53        1.06          -0.47    -30.48%       33.5%        7.9%       58.6%   ⭐⭐ (p<0.01)
                home_timespent         231 (3.9%)       257.78s      69.04s       -188.74s    -73.22%       39.8%        0.0%       60.2%            NS
               price_timespent         122 (2.1%)        96.30s     121.11s        +24.81s    +25.76%       52.5%        0.0%       47.5%            NS
           

In [869]:
import pandas as pd
import numpy as np

print("=" * 125)
print("🚀 UNDILUTED ACTIVE-PAIR TRANSITION MATRIX (EXCLUDES 0-0=0 INACTIVE PAIRS)")
print("=" * 125)

df_paired = df_paired_episodes.copy()

# 1. Label Transition Pairs (2->1, 3->2, 4->3, 5+->4+)
def label_transition_pair(r):
    curr = int(r.get('episode_number_curr', 2))
    prev = int(r.get('episode_number_prev', 1))
    if curr <= 4:
        return f"{curr} ➔ {prev}"
    else:
        return "5+ ➔ 4+"

df_paired['transition_pair'] = df_paired.apply(label_transition_pair, axis=1)

# 2. Key metrics to analyze
key_metrics = [
    'total_web_timespent', 'avg_web_activity', 'active_session_count', 'total_session_count',
    'home_timespent', 'price_timespent', 'plan_timespent', 'location_timespent',
    'amenities_timespent', 'media_timespent', 'specifications_timespent',
    'total_answered_calls', 'total_inbound_calls', 'total_outbound_calls',
    'total_agent_missed_calls', 'total_customer_missed_calls', 'total_attempted_calls',
    'total_call_duration',
    'episode_whatsapp_inbound_count', 'resubmission_count_in_episode',
    'episode_stage_movement_count', 'episode_duration_days'
]

transition_results = []

for pair_name in ['2 ➔ 1', '3 ➔ 2', '4 ➔ 3', '5+ ➔ 4+']:
    sub_df = df_paired[df_paired['transition_pair'] == pair_name]
    if len(sub_df) == 0:
        continue
        
    pair_dict = {'Transition Pair': pair_name, 'Total Pairs': len(sub_df)}
    
    for m in key_metrics:
        m_prev_col = f"{m}_prev"
        m_curr_col = f"{m}_curr"
        
        if m_prev_col in sub_df.columns and m_curr_col in sub_df.columns:
            v_prev = pd.to_numeric(sub_df[m_prev_col], errors='coerce').fillna(0.0)
            v_curr = pd.to_numeric(sub_df[m_curr_col], errors='coerce').fillna(0.0)
            
            # Active-Pair Filter: Prev > 0 OR Curr > 0 (for days, >= 0)
            if "days" in m:
                active_mask = (v_prev >= 0) | (v_curr >= 0)
            else:
                active_mask = (v_prev > 0) | (v_curr > 0)
                
            v_prev_act = v_prev[active_mask]
            v_curr_act = v_curr[active_mask]
            
            n_act = len(v_prev_act)
            
            if n_act > 0:
                mean_prev = v_prev_act.mean()
                mean_curr = v_curr_act.mean()
                delta_mean = mean_curr - mean_prev
                
                unit = "s" if "timespent" in m or m in ["total_call_duration", "avg_web_activity"] else ("d" if "days" in m else "")
                
                # Format: Current Avg (Mean Shift Δ) [Active n]
                pair_dict[m] = f"{mean_curr:.1f}{unit} ({delta_mean:+.1f}{unit}) [n={n_act}]"
            else:
                pair_dict[m] = "N/A"
                
    transition_results.append(pair_dict)

df_active_transition_report = pd.DataFrame(transition_results)

print("\n📊 UNDILUTED TRANSITION MATRIX [Format: Current Active Avg (Mean Shift Δ) [Active n]]:")
print("-" * 125)
print(df_active_transition_report.to_string(index=False))


🚀 UNDILUTED ACTIVE-PAIR TRANSITION MATRIX (EXCLUDES 0-0=0 INACTIVE PAIRS)

📊 UNDILUTED TRANSITION MATRIX [Format: Current Active Avg (Mean Shift Δ) [Active n]]:
-----------------------------------------------------------------------------------------------------------------------------
Transition Pair  Total Pairs      total_web_timespent active_session_count total_session_count          home_timespent         price_timespent          plan_timespent   location_timespent  amenities_timespent       media_timespent specifications_timespent total_answered_calls total_inbound_calls total_outbound_calls total_agent_missed_calls total_customer_missed_calls total_attempted_calls     total_call_duration episode_whatsapp_inbound_count resubmission_count_in_episode episode_stage_movement_count  episode_duration_days
          2 ➔ 1         5023 171.5s (-220.3s) [n=301]   0.9 (-0.3) [n=301]  1.0 (-0.5) [n=369] 66.4s (-201.5s) [n=212] 127.9s (+45.3s) [n=108] 144.9s (-372.8s) [n=89] 28.5s (+1.4s) [n

,lead,episode_number,episode_start_time,episode_end_time,start_stage,end_stage,total_stage_movements,episode_id,is_reactivated_episode,masterLeadId,bornSource,source,secondarySource,project,status,lead_createdAt,globalId,leadId,visitorId,total_web_timespent,active_session_count,home_timespent,price_timespent,plan_timespent,location_timespent,amenities_timespent,media_timespent,specifications_timespent,is_site_visit_book_clicked,is_whatsapp_clicked,is_brochure_downloaded,is_price_sheet_viewed,is_cost_sheet_clicked,total_session_count,resubmission_count_in_episode,is_resubmitted_in_episode,did_reach_site_visit_in_episode,episode_stage_movement_count,stage_before_ni,cancellation_reason_at_episode_end,episode_duration_days,episode_source_id,episode_source_name,episode_whatsapp_inbound_count,has_whatsapp_inbound_in_episode,days_to_first_whatsapp_inbound,days_to_last_whatsapp_inbound,total_answered_calls,total_inbound_calls,total_outbound_calls,total_agent_missed_calls,total_customer_missed_calls,total_attempted_calls,total_call_duration,pickup_ratio
0,6a023a53d90c4857b766189c,1,2026-05-11 20:21:39.744,2026-05-17 10:07:15.379,New Lead,Not Interested,15,6a023a53d90c4857b766189c_1,0,920554,64a7859a0fcd685ef70d6dd8,64a7859a0fcd685ef70d6dd8,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:21:39.461,22787547,1177568,Visitor-26920396,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,64a785845bbdfeeb1a2372cf,5.573329,64a7859a0fcd685ef70d6dd8,Web Lead,0,0,-1.0,-1.0,0,0,10,4,6,6,0.000,0.000000
1,6a023b888b2d7b451592811f,1,2026-05-11 20:26:49.143,2026-05-11 21:14:33.150,New Lead,Not Interested,2,6a023b888b2d7b451592811f_1,0,920555,66d6b121137a13b8864d16a2,66d6b121137a13b8864d16a2,[],64a5178ed9039b2988d6e4ab,<NA>,2026-05-11 20:26:48.757,<NA>,1177569,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0,0,0,2,New Lead,65bb96cf3860e75ad7870153,0.033148,66d6b121137a13b8864d16a2,FIM Forms,0,0,-1.0,-1.0,1,0,1,0,0,1,46.608,100.000000
2,6a023bb0f858b6e164691c23,1,2026-05-11 20:27:28.906,2026-05-25 06:44:24.791,New Lead,Not Interested,9,6a023bb0f858b6e164691c23_1,0,920556,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:27:28.565,<NA>,1177570,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad787015a,13.428425,69d8ae93e2ee6bc0cbfe6674,Pre-launch Web,0,0,-1.0,-1.0,4,0,6,2,0,4,661.214,100.000000
3,6a023c83ddaa0af3065f4e91,1,2026-05-11 20:30:59.685,2026-05-12 09:42:39.472,New Lead,Not Interested,7,6a023c83ddaa0af3065f4e91_1,0,920557,69d8ae93e2ee6bc0cbfe6674,69d8ae93e2ee6bc0cbfe6674,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:30:59.331,<NA>,1177571,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad7870153,0.549766,69d8ae93e2ee6bc0cbfe6674,Pre-launch Web,0,0,-1.0,-1.0,1,0,2,1,0,1,109.904,100.000000
4,6a023d25d90c4857b7661b5d,1,2026-05-11 20:33:41.442,2026-05-15 11:00:03.718,New Lead,Not Interested,10,6a023d25d90c4857b7661b5d_1,0,801227,69d4979d60cfeeb29bcd905b,69d4979d60cfeeb29bcd905b,[],68f228762292e21cc6165c2e,<NA>,2026-05-11 20:33:41.149,<NA>,1177572,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0,0,0,3,Lead Initiated,65bb96cf3860e75ad787015c,3.601647,69d4979d60cfeeb29bcd905b,Whatsapp,0,0,-1.0,-1.0,1,0,6,3,2,3,11.000,33.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111733,6a792178a03a0a265480d9d7,1,2026-08-10 00:55:20.575,2026-08-10 00:55:20.575,New Lead,New Lead,1,6a792178a03a0a265480d9d7_1,0,1004566,66683e8865601d05fa511e02,66683e8865601d05fa511e02,[],65adfbb3f09468f0e32fba49,<NA>,2026-08-10 00:55:20.244,23831789,1283426,Visitor-27976922,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0,0,0,1,New Lead,None,0.000000,66683e8865601d05fa511e02,Web Lead,0,0,-1.0,-1.0,0,0,0,0,0,0,0.000,NaN
111734,6a7921f